# First-Order Greeks: Delta, Vega, Theta, and Rho

**Level:** Intermediate
**Concepts Covered:** 7

---

## Overview

This comprehensive notebook covers first-order Greeks (Delta, Vega, Theta, Rho) with detailed explanations, mathematical derivations, Python implementations, and practical examples. You'll learn how to calculate, visualize, and apply these essential risk metrics to manage options portfolios effectively.

## 1. Introduction

### Why Greeks Matter

**Option Greeks** are the foundation of derivatives risk management and trading. They measure how option prices change in response to market conditions, enabling traders to:

- **Hedge risk** effectively by understanding exposure to different market factors
- **Price options** more accurately by decomposing price sensitivity
- **Manage portfolios** of complex derivatives positions
- **Execute strategies** that profit from specific market movements

### Real-World Applications

- **Market Making**: Delta-neutral market makers use Greeks to hedge inventory risk
- **Portfolio Management**: Fund managers use Greeks to control overall risk exposure
- **Regulatory Compliance**: Basel III requires banks to measure and report Greek exposures
- **Algorithmic Trading**: High-frequency traders use Greeks for automated hedging
- **Risk Analytics**: Risk departments monitor Greeks across entire trading books

### Learning Objectives

By the end of this notebook, you will be able to:

1. **Calculate** Delta analytically and numerically for European options
2. **Implement** delta-hedging strategies and understand rebalancing frequency
3. **Compute** Vega to measure volatility sensitivity and manage vol risk
4. **Calculate** Theta to quantify time decay and understand theta-gamma relationship
5. **Determine** Rho to assess interest rate sensitivity
6. **Apply** Greeks to real-world portfolio risk management scenarios

### Prerequisites

- Understanding of Black-Scholes option pricing model
- Familiarity with partial derivatives (calculus)
- Python programming with NumPy and Matplotlib
- Basic knowledge of option payoffs and strategies

### Historical Context

The concept of Greeks emerged from the Black-Scholes-Merton framework (1973), which provided closed-form formulas for option sensitivities. The term "Greeks" became standard because:

- Most sensitivities use Greek letters: Δ (Delta), Γ (Gamma), Θ (Theta), ν (Vega), ρ (Rho)
- Vega is not actually a Greek letter (it's ν or sometimes ν is used)
- These measures became industry standard for risk management in the 1980s
- Modern derivatives desks monitor hundreds of Greeks across portfolios

**Estimated Time**: 90-120 minutes

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, optimize
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

# Set random seed for reproducibility
np.random.seed(42)

print('[OK] Libraries imported successfully')

## 2. Delta: Price Sensitivity

### Theory

**Delta (Δ)** measures the rate of change of an option's price with respect to changes in the underlying asset price:

$$
\Delta = \frac{\partial V}{\partial S}
$$

where $V$ is the option value and $S$ is the stock price.

### Key Properties of Delta

**Range of Values**:
- **Call options**: $0 \leq \Delta_{\text{call}} \leq 1$
- **Put options**: $-1 \leq \Delta_{\text{put}} \leq 0$
- **At-the-money (ATM)**: $\Delta_{\text{call}} \approx 0.5$, $\Delta_{\text{put}} \approx -0.5$
- **Deep in-the-money (ITM)**: $\Delta_{\text{call}} \to 1$, $\Delta_{\text{put}} \to -1$
- **Deep out-of-the-money (OTM)**: $\Delta_{\text{call}} \to 0$, $\Delta_{\text{put}} \to 0$

### Financial Interpretation

**Hedge Ratio**: Delta represents the number of shares needed to hedge one option:
- If $\Delta = 0.6$, hold 0.6 shares per long call option to achieve delta-neutral position
- Delta changes as stock price and time change, requiring rebalancing

**Probability Approximation**: Delta ≈ Risk-neutral probability of finishing in-the-money
- ATM option with $\Delta = 0.5$ has ~50% chance of finishing ITM
- This is an approximation; actual probability differs slightly

**Leverage**: Delta measures the option's leverage relative to the stock:
- Option with $\Delta = 0.8$ moves $0.80 for every $1 move in stock
- Provides exposure without full capital outlay

### Mathematical Formulation

For European options under Black-Scholes model, Delta has closed-form expressions:

**Call Option Delta**:
$$
\Delta_{\text{call}} = N(d_1)
$$

**Put Option Delta**:
$$
\Delta_{\text{put}} = N(d_1) - 1 = -N(-d_1)
$$

where:

$$
d_1 = \frac{\ln(S/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}
$$

and $N(\cdot)$ is the cumulative standard normal distribution function.

### Put-Call Parity for Delta

Delta satisfies put-call parity:
$$
\Delta_{\text{call}} - \Delta_{\text{put}} = 1
$$

This follows from the put-call parity relationship for option prices.

### Delta as a Function of Moneyness

Delta varies with the moneyness ratio $S/K$:

- **Deep ITM** ($S/K \gg 1$ for calls): $\Delta_{\text{call}} \to 1$ (behaves like stock)
- **ATM** ($S/K = 1$): $\Delta_{\text{call}} \approx 0.5$ (50% sensitivity)
- **Deep OTM** ($S/K \ll 1$ for calls): $\Delta_{\text{call}} \to 0$ (minimal sensitivity)

### Numerical Approximation

Delta can be approximated using finite differences:
$$
\Delta \approx \frac{V(S + \Delta S) - V(S - \Delta S)}{2\Delta S}
$$

This is useful when closed-form formulas are unavailable (e.g., exotic options).

In [ ]:
# Black-Scholes Greeks Implementation

def black_scholes_price(S, K, T, r, sigma, option_type='call'):
    """
    Calculate Black-Scholes option price.
    
    Parameters
    ----------
    S : float
        Current stock price
    K : float
        Strike price
    T : float
        Time to expiration (years)
    r : float
        Risk-free rate
    sigma : float
        Volatility (annualized)
    option_type : str
        'call' or 'put'
    
    Returns
    -------
    float
        Option price
    """
    if T <= 0:
        if option_type == 'call':
            return max(S - K, 0)
        else:
            return max(K - S, 0)
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    if option_type == 'call':
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    
    return price


def calculate_delta(S, K, T, r, sigma, option_type='call'):
    """
    Calculate option Delta using Black-Scholes formula.
    
    Parameters
    ----------
    S : float
        Current stock price
    K : float
        Strike price
    T : float
        Time to expiration (years)
    r : float
        Risk-free rate
    sigma : float
        Volatility (annualized)
    option_type : str
        'call' or 'put'
    
    Returns
    -------
    float
        Option Delta
    """
    if T <= 0:
        if option_type == 'call':
            return 1.0 if S > K else 0.0
        else:
            return -1.0 if S < K else 0.0
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    
    if option_type == 'call':
        delta = norm.cdf(d1)
    else:
        delta = norm.cdf(d1) - 1
    
    return delta


def calculate_delta_numerical(S, K, T, r, sigma, option_type='call', dS=0.01):
    """
    Calculate option Delta using numerical finite difference.
    
    Parameters
    ----------
    S : float
        Current stock price
    K : float
        Strike price
    T : float
        Time to expiration (years)
    r : float
        Risk-free rate
    sigma : float
        Volatility (annualized)
    option_type : str
        'call' or 'put'
    dS : float
        Small change in stock price for finite difference
    
    Returns
    -------
    float
        Option Delta (numerical approximation)
    """
    V_up = black_scholes_price(S + dS, K, T, r, sigma, option_type)
    V_down = black_scholes_price(S - dS, K, T, r, sigma, option_type)
    
    delta = (V_up - V_down) / (2 * dS)
    return delta


# Example: Calculate Delta for ATM call option
print("=" * 70)
print("DELTA CALCULATION: AT-THE-MONEY CALL OPTION")
print("=" * 70)

S = 100
K = 100
T = 0.25  # 3 months
r = 0.05
sigma = 0.20

delta_analytical = calculate_delta(S, K, T, r, sigma, option_type='call')
delta_numerical = calculate_delta_numerical(S, K, T, r, sigma, option_type='call')
option_price = black_scholes_price(S, K, T, r, sigma, option_type='call')

print(f"\nOption Parameters:")
print(f"  Stock Price (S): ${S:.2f}")
print(f"  Strike Price (K): ${K:.2f} (ATM)")
print(f"  Time to Expiration: {T*252:.0f} trading days ({T*12:.1f} months)")
print(f"  Risk-Free Rate: {r:.2%}")
print(f"  Volatility: {sigma:.1%}")

print(f"\nOption Price: ${option_price:.4f}")

print(f"\nDelta Calculations:")
print(f"  Analytical (Black-Scholes): {delta_analytical:.6f}")
print(f"  Numerical (Finite Difference): {delta_numerical:.6f}")
print(f"  Difference: {abs(delta_analytical - delta_numerical):.8f}")

print(f"\nInterpretation:")
print(f"  • Delta = {delta_analytical:.4f} means:")
print(f"    - Option price changes by ${delta_analytical:.4f} for $1 change in stock")
print(f"    - Hold {delta_analytical:.4f} shares to delta-hedge 1 call option")
print(f"    - ~{delta_analytical*100:.1f}% probability of finishing in-the-money")

print(f"\nHedging Example (100 call contracts):")
shares_needed = delta_analytical * 100 * 100  # 100 contracts * 100 shares/contract
print(f"  • Long 100 call contracts (Delta = {delta_analytical:.4f} each)")
print(f"  • Total portfolio Delta: {delta_analytical * 100:.2f}")
print(f"  • To delta-hedge: Short {shares_needed:.0f} shares")
print(f"  • Hedge value: ${shares_needed * S:,.2f}")

print("\n" + "=" * 70)
print('[OK] Delta calculation implementation complete')

In [ ]:
# Visualization: Delta vs Stock Price and Strike

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

# Parameters
S_range = np.linspace(70, 130, 100)
K = 100
T = 0.25
r = 0.05
sigma = 0.20

# Plot 1: Delta vs Stock Price (Fixed Strike)
deltas_call = [calculate_delta(S, K, T, r, sigma, 'call') for S in S_range]
deltas_put = [calculate_delta(S, K, T, r, sigma, 'put') for S in S_range]

ax1.plot(S_range, deltas_call, 'b-', linewidth=2, label='Call Delta')
ax1.plot(S_range, deltas_put, 'r-', linewidth=2, label='Put Delta')
ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax1.axhline(y=0.5, color='green', linestyle='--', linewidth=1, alpha=0.5)
ax1.axhline(y=-0.5, color='orange', linestyle='--', linewidth=1, alpha=0.5)
ax1.axvline(x=K, color='gray', linestyle='--', linewidth=1.5, label=f'Strike K=${K}')

ax1.set_xlabel('Stock Price ($)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Delta', fontsize=12, fontweight='bold')
ax1.set_title('Delta vs Stock Price (K=$100, T=3mo)', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(loc='best', fontsize=10)
ax1.set_ylim([-1.1, 1.1])

# Add annotations
ax1.annotate('Deep ITM\nCall: Δ→1', xy=(125, 0.95), fontsize=9, 
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
ax1.annotate('Deep OTM\nCall: Δ→0', xy=(75, 0.05), fontsize=9,
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))

# Plot 2: Delta vs Time to Expiration
T_range = np.linspace(0.01, 1.0, 100)
S_scenarios = [90, 100, 110]

for S_val in S_scenarios:
    deltas = [calculate_delta(S_val, K, T_val, r, sigma, 'call') for T_val in T_range]
    label = f'S=${S_val} ({"OTM" if S_val < K else "ATM" if S_val == K else "ITM"})'
    ax2.plot(T_range, deltas, linewidth=2, label=label)

ax2.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax2.set_xlabel('Time to Expiration (years)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Call Delta', fontsize=12, fontweight='bold')
ax2.set_title('Call Delta vs Time for Different Moneyness', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend(loc='best', fontsize=10)

# Plot 3: Delta vs Strike Price
K_range = np.linspace(70, 130, 100)
S = 100

deltas_call_k = [calculate_delta(S, K_val, T, r, sigma, 'call') for K_val in K_range]
deltas_put_k = [calculate_delta(S, K_val, T, r, sigma, 'put') for K_val in K_range]

ax3.plot(K_range, deltas_call_k, 'b-', linewidth=2, label='Call Delta')
ax3.plot(K_range, deltas_put_k, 'r-', linewidth=2, label='Put Delta')
ax3.axvline(x=S, color='green', linestyle='--', linewidth=1.5, label=f'Stock S=${S}')
ax3.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

ax3.set_xlabel('Strike Price ($)', fontsize=12, fontweight='bold')
ax3.set_ylabel('Delta', fontsize=12, fontweight='bold')
ax3.set_title('Delta vs Strike Price (S=$100, T=3mo)', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.legend(loc='best', fontsize=10)
ax3.set_ylim([-1.1, 1.1])

# Plot 4: Delta vs Volatility
sigma_range = np.linspace(0.10, 0.50, 100)
S = 100
K = 100

deltas_vol = [calculate_delta(S, K, T, r, sig, 'call') for sig in sigma_range]

ax4.plot(sigma_range * 100, deltas_vol, 'purple', linewidth=2)
ax4.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Δ = 0.5')
ax4.set_xlabel('Volatility (%)', fontsize=12, fontweight='bold')
ax4.set_ylabel('ATM Call Delta', fontsize=12, fontweight='bold')
ax4.set_title('ATM Call Delta vs Volatility', fontsize=13, fontweight='bold')
ax4.grid(True, alpha=0.3)
ax4.legend(loc='best', fontsize=10)

# Add text annotation
ax4.text(0.05, 0.95, f'S=K=${K}\nT={T} years\nr={r:.1%}',
        transform=ax4.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print('[OK] Delta visualization complete')

### Practical Example\n\nLet's apply delta: price sensitivity to a real-world scenario...

In [ ]:
# Practical Example: Delta Across Moneyness

print("=" * 80)
print("DELTA ANALYSIS: COMPARING CALLS AND PUTS ACROSS MONEYNESS")
print("=" * 80)

S = 100
T = 0.25
r = 0.05
sigma = 0.20

strikes = [80, 90, 95, 100, 105, 110, 120]
moneyness_labels = ['Deep ITM', 'ITM', 'Slightly ITM', 'ATM', 'Slightly OTM', 'OTM', 'Deep OTM']

print(f"\nStock Price: ${S:.2f}")
print(f"Time to Expiration: {T*12:.1f} months")
print(f"Volatility: {sigma:.1%}")
print(f"\n{'Strike':<10} {'Moneyness':<15} {'Call Δ':<12} {'Put Δ':<12} {'Δ(Call) - Δ(Put)':<18}")
print("-" * 80)

for i, K in enumerate(strikes):
    delta_call = calculate_delta(S, K, T, r, sigma, 'call')
    delta_put = calculate_delta(S, K, T, r, sigma, 'put')
    call_price = black_scholes_price(S, K, T, r, sigma, 'call')
    put_price = black_scholes_price(S, K, T, r, sigma, 'put')
    
    print(f"${K:<9.0f} {moneyness_labels[i]:<15} {delta_call:>11.6f} {delta_put:>11.6f} {delta_call - delta_put:>17.6f}")

print(f"\n\nKey Observations:")
print(f"  1. Call Delta ranges from ~0 (deep OTM) to ~1 (deep ITM)")
print(f"  2. Put Delta ranges from ~-1 (deep ITM) to ~0 (deep OTM)")
print(f"  3. Delta(Call) - Delta(Put) ≈ 1 for all strikes (put-call parity)")
print(f"  4. ATM options have |Delta| ≈ 0.5")

print(f"\n\nPractical Implications:")
print(f"  • Deep ITM calls behave like stock (Δ ≈ 1)")
print(f"  • Deep OTM calls have minimal price sensitivity (Δ ≈ 0)")
print(f"  • ATM options provide balanced leverage (Δ ≈ 0.5)")
print(f"  • Put Delta is negative (inverse relationship with stock price)")

print("\n" + "=" * 80)
print('[OK] Delta moneyness analysis complete')

## 3. Delta Hedging Strategies

### Theory

**Delta hedging** is the practice of offsetting the directional risk (delta) of an option position by taking an opposite position in the underlying asset.

### The Delta-Neutral Portfolio

A **delta-neutral** portfolio has zero total delta:
$$
\Delta_{\text{portfolio}} = \sum_{i=1}^{n} w_i \Delta_i = 0
$$

where $w_i$ is the quantity of instrument $i$ with delta $\Delta_i$.

### Hedging Mechanics

**For a Long Call Position**:
- Long 1 call with $\Delta = 0.6$
- To hedge: Short $0.6$ shares
- Net delta: $+0.6 - 0.6 = 0$

**For a Short Put Position**:
- Short 1 put with $\Delta = -0.4$
- Position delta: $-(-0.4) = +0.4$
- To hedge: Short $0.4$ shares

### Why Delta Hedging?

1. **Market Making**: Market makers hedge inventory to profit from bid-ask spread
2. **Volatility Trading**: Isolate volatility (vega) exposure from directional (delta) risk
3. **Risk Management**: Reduce sensitivity to stock price movements
4. **Arbitrage**: Lock in mispricings while hedging directional risk

### Dynamic Rehedging

Delta changes as the stock price and time change (**gamma** and **theta** effects), requiring periodic rebalancing:

- **Continuous hedging** (theoretical): Infinitely frequent rebalancing
- **Practical hedging**: Rebalance daily, hourly, or when delta drifts beyond threshold
- **Transaction costs**: Trade-off between hedge accuracy and trading costs

### Mathematical Formulation

The delta-hedged portfolio consists of an option position and a hedge position in the underlying:

$$
\Pi = V(S, t) - \Delta \cdot S
$$

where $\Pi$ is the portfolio value, $V$ is the option value, and $\Delta$ shares are held short.

**Self-Financing Condition**:

For a delta-neutral portfolio, the change in portfolio value is:

$$
d\Pi = dV - \Delta \cdot dS
$$

Using Ito's lemma to expand $dV$:

$$
dV = \frac{\partial V}{\partial S}dS + \frac{\partial V}{\partial t}dt + \frac{1}{2}\frac{\partial^2 V}{\partial S^2}(dS)^2
$$

With $dS = \mu S dt + \sigma S dW$ (geometric Brownian motion), we get:

$$
d\Pi = \left(\frac{\partial V}{\partial t} + \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2}\right)dt + \left(\frac{\partial V}{\partial S} - \Delta\right)dS
$$

**Delta-Neutral Condition**: Setting $\Delta = \frac{\partial V}{\partial S}$ eliminates the stochastic $dS$ term:

$$
d\Pi = \left(\frac{\partial V}{\partial t} + \frac{1}{2}\sigma^2 S^2 \Gamma\right)dt
$$

This shows that a delta-hedged portfolio has instantaneous P&L driven by **theta** and **gamma**.

**Rehedging Frequency**:

The hedge ratio must be updated as delta changes:
- **Continuous hedging**: Theoretical ideal, impractical due to infinite transaction costs
- **Discrete hedging**: Rebalance at intervals $\Delta t$
- **Threshold hedging**: Rebalance when delta drift exceeds tolerance

**Transaction Costs**:

Total transaction cost for $N$ rehedges over lifetime:
$$
C_{\text{total}} = \sum_{i=1}^{N} c \cdot |\Delta S_i|
$$

where $c$ is the proportional transaction cost and $|\Delta S_i|$ is the absolute change in hedge.

In [ ]:
# Delta Hedging Simulation with Dynamic Rebalancing

def calculate_gamma(S, K, T, r, sigma):
    """
    Calculate option Gamma using Black-Scholes formula.
    
    Gamma is the same for calls and puts.
    
    Parameters
    ----------
    S : float
        Current stock price
    K : float
        Strike price
    T : float
        Time to expiration (years)
    r : float
        Risk-free rate
    sigma : float
        Volatility (annualized)
    
    Returns
    -------
    float
        Option Gamma
    """
    if T <= 0:
        return 0.0
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    
    return gamma


def simulate_delta_hedging(S0, K, T, r, sigma, option_type='call', 
                          n_steps=50, rehedge_freq='daily', 
                          transaction_cost=0.001, n_paths=1):
    """
    Simulate dynamic delta hedging strategy with transaction costs.
    
    Parameters
    ----------
    S0 : float
        Initial stock price
    K : float
        Strike price
    T : float
        Time to expiration (years)
    r : float
        Risk-free rate
    sigma : float
        Volatility (annualized)
    option_type : str
        'call' or 'put'
    n_steps : int
        Number of time steps
    rehedge_freq : str or int
        'daily' or integer for steps between rehedges
    transaction_cost : float
        Proportional transaction cost
    n_paths : int
        Number of simulation paths
    
    Returns
    -------
    dict
        Dictionary containing simulation results
    """
    dt = T / n_steps
    
    # Determine rehedging frequency
    if rehedge_freq == 'daily':
        rehedge_interval = 1
    else:
        rehedge_interval = int(rehedge_freq)
    
    # Storage for results
    paths = np.zeros((n_paths, n_steps + 1))
    portfolio_values = np.zeros((n_paths, n_steps + 1))
    deltas = np.zeros((n_paths, n_steps + 1))
    hedge_positions = np.zeros((n_paths, n_steps + 1))
    transaction_costs = np.zeros((n_paths, n_steps + 1))
    
    for path in range(n_paths):
        S = S0
        hedge_shares = 0
        cumulative_cost = 0
        
        for step in range(n_steps + 1):
            time_to_expiry = T - step * dt
            
            # Calculate current option value and delta
            if time_to_expiry > 0:
                option_value = black_scholes_price(S, K, time_to_expiry, r, sigma, option_type)
                delta = calculate_delta(S, K, time_to_expiry, r, sigma, option_type)
            else:
                # At expiration
                if option_type == 'call':
                    option_value = max(S - K, 0)
                else:
                    option_value = max(K - S, 0)
                delta = 0
            
            # Rehedge if it's time to rebalance
            if step % rehedge_interval == 0 and time_to_expiry > 0:
                shares_to_trade = delta - hedge_shares
                cost = abs(shares_to_trade) * S * transaction_cost
                cumulative_cost += cost
                hedge_shares = delta
                transaction_costs[path, step] = cost
            
            # Portfolio value: long option + short delta shares
            portfolio_value = option_value - hedge_shares * S
            
            # Store values
            paths[path, step] = S
            portfolio_values[path, step] = portfolio_value
            deltas[path, step] = delta
            hedge_positions[path, step] = hedge_shares
            
            # Simulate next stock price
            if step < n_steps:
                Z = np.random.standard_normal()
                S = S * np.exp((r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z)
    
    return {
        'stock_paths': paths,
        'portfolio_values': portfolio_values,
        'deltas': deltas,
        'hedge_positions': hedge_positions,
        'transaction_costs': transaction_costs,
        'time_grid': np.linspace(0, T, n_steps + 1)
    }


# Example: Simulate Delta Hedging for ATM Call Option
print("=" * 80)
print("DELTA HEDGING SIMULATION: DYNAMIC REBALANCING")
print("=" * 80)

S0 = 100
K = 100
T = 0.25  # 3 months
r = 0.05
sigma = 0.20
transaction_cost = 0.001  # 10 bps

print(f"\nSimulation Parameters:")
print(f"  Initial Stock Price: ${S0:.2f}")
print(f"  Strike Price: ${K:.2f} (ATM)")
print(f"  Time to Expiration: {T*252:.0f} trading days")
print(f"  Volatility: {sigma:.1%}")
print(f"  Transaction Cost: {transaction_cost:.2%}")

# Simulate with daily rehedging
results_daily = simulate_delta_hedging(S0, K, T, r, sigma, option_type='call',
                                       n_steps=63, rehedge_freq='daily',
                                       transaction_cost=transaction_cost, n_paths=1)

# Simulate with weekly rehedging
results_weekly = simulate_delta_hedging(S0, K, T, r, sigma, option_type='call',
                                        n_steps=63, rehedge_freq=5,
                                        transaction_cost=transaction_cost, n_paths=1)

# Calculate statistics
initial_option_price = black_scholes_price(S0, K, T, r, sigma, 'call')
final_option_value_daily = results_daily['portfolio_values'][0, -1]
final_option_value_weekly = results_weekly['portfolio_values'][0, -1]

total_cost_daily = np.sum(results_daily['transaction_costs'][0])
total_cost_weekly = np.sum(results_weekly['transaction_costs'][0])

print(f"\nInitial Option Price: ${initial_option_price:.4f}")
print(f"\nDaily Rehedging Results:")
print(f"  Number of Rehedges: {np.sum(results_daily['transaction_costs'][0] > 0):.0f}")
print(f"  Total Transaction Costs: ${total_cost_daily:.4f}")
print(f"  Final Portfolio Value: ${final_option_value_daily:.4f}")
print(f"  Final Stock Price: ${results_daily['stock_paths'][0, -1]:.2f}")

print(f"\nWeekly Rehedging Results:")
print(f"  Number of Rehedges: {np.sum(results_weekly['transaction_costs'][0] > 0):.0f}")
print(f"  Total Transaction Costs: ${total_cost_weekly:.4f}")
print(f"  Final Portfolio Value: ${final_option_value_weekly:.4f}")
print(f"  Final Stock Price: ${results_weekly['stock_paths'][0, -1]:.2f}")

print(f"\nKey Insights:")
print(f"  • Daily rehedging: More accurate hedge, but higher transaction costs")
print(f"  • Weekly rehedging: Lower costs, but larger tracking error")
print(f"  • Trade-off: Hedge accuracy vs. transaction costs")

print("\n" + "=" * 80)
print('[OK] Delta hedging simulation complete')

In [ ]:
# Visualization: Delta Hedging Performance Over Time

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# Use the simulation results from previous cell
time_grid = results_daily['time_grid'] * 252  # Convert to trading days

# Plot 1: Stock Price Path
ax1.plot(time_grid, results_daily['stock_paths'][0], 'b-', linewidth=2, label='Stock Price')
ax1.axhline(y=K, color='red', linestyle='--', linewidth=1.5, label=f'Strike K=${K}')
ax1.fill_between(time_grid, K, results_daily['stock_paths'][0], 
                  where=(results_daily['stock_paths'][0] >= K), alpha=0.3, 
                  color='green', label='In-the-Money')
ax1.set_xlabel('Time (trading days)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Stock Price ($)', fontsize=12, fontweight='bold')
ax1.set_title('Stock Price Evolution During Delta Hedge', fontsize=13, fontweight='bold')
ax1.legend(loc='best', fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 2: Delta Evolution Over Time
ax2.plot(time_grid, results_daily['deltas'][0], 'purple', linewidth=2, label='Option Delta')
ax2.plot(time_grid, results_daily['hedge_positions'][0], 'orange', 
         linewidth=2, linestyle='--', label='Hedge Position')
ax2.axhline(y=0.5, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax2.set_xlabel('Time (trading days)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Delta / Hedge Shares', fontsize=12, fontweight='bold')
ax2.set_title('Delta and Hedge Position Over Time (Daily Rehedging)', fontsize=13, fontweight='bold')
ax2.legend(loc='best', fontsize=10)
ax2.grid(True, alpha=0.3)

# Plot 3: Portfolio Value Comparison
ax3.plot(time_grid, results_daily['portfolio_values'][0], 'b-', 
         linewidth=2, label='Daily Rehedging')
ax3.plot(time_grid, results_weekly['portfolio_values'][0], 'r--', 
         linewidth=2, label='Weekly Rehedging')
ax3.set_xlabel('Time (trading days)', fontsize=12, fontweight='bold')
ax3.set_ylabel('Portfolio Value ($)', fontsize=12, fontweight='bold')
ax3.set_title('Delta-Hedged Portfolio Value', fontsize=13, fontweight='bold')
ax3.legend(loc='best', fontsize=10)
ax3.grid(True, alpha=0.3)

# Add text annotation
final_pnl_daily = results_daily['portfolio_values'][0, -1] - results_daily['portfolio_values'][0, 0]
final_pnl_weekly = results_weekly['portfolio_values'][0, -1] - results_weekly['portfolio_values'][0, 0]
ax3.text(0.02, 0.98, f'Daily PnL: ${final_pnl_daily:.4f}\nWeekly PnL: ${final_pnl_weekly:.4f}',
        transform=ax3.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# Plot 4: Cumulative Transaction Costs
cumulative_cost_daily = np.cumsum(results_daily['transaction_costs'][0])
cumulative_cost_weekly = np.cumsum(results_weekly['transaction_costs'][0])

ax4.plot(time_grid, cumulative_cost_daily, 'b-', linewidth=2, label='Daily Rehedging')
ax4.plot(time_grid, cumulative_cost_weekly, 'r--', linewidth=2, label='Weekly Rehedging')
ax4.set_xlabel('Time (trading days)', fontsize=12, fontweight='bold')
ax4.set_ylabel('Cumulative Transaction Costs ($)', fontsize=12, fontweight='bold')
ax4.set_title('Transaction Costs: Daily vs Weekly Rehedging', fontsize=13, fontweight='bold')
ax4.legend(loc='best', fontsize=10)
ax4.grid(True, alpha=0.3)

# Add annotation
total_cost_daily = cumulative_cost_daily[-1]
total_cost_weekly = cumulative_cost_weekly[-1]
ax4.text(0.98, 0.02, f'Daily Total: ${total_cost_daily:.4f}\nWeekly Total: ${total_cost_weekly:.4f}',
        transform=ax4.transAxes, fontsize=10, verticalalignment='bottom',
        horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

plt.tight_layout()
plt.show()

print('[OK] Delta hedging visualization complete')

### Practical Example: Market Maker Delta Hedging

Let's simulate a real-world market making scenario where a trader sells options and must delta-hedge continuously.

In [ ]:
# Practical Example: Market Maker Sells 1000 Call Options

print("=" * 80)
print("PRACTICAL EXAMPLE: MARKET MAKER DELTA HEDGING")
print("=" * 80)

# Market maker sells 1000 ATM call options
n_contracts = 1000
contract_size = 100  # shares per contract

S0 = 100
K = 100
T = 0.5  # 6 months
r = 0.05
sigma = 0.25
premium_received = 8.50  # per share

print(f"\nScenario: Market Maker Sells {n_contracts} Call Option Contracts")
print(f"  Stock Price: ${S0:.2f}")
print(f"  Strike: ${K:.2f}")
print(f"  Expiration: {T*12:.0f} months")
print(f"  Implied Volatility: {sigma:.1%}")
print(f"  Premium Received: ${premium_received:.2f} per share")

# Calculate initial option value and delta
initial_option_value = black_scholes_price(S0, K, T, r, sigma, 'call')
initial_delta = calculate_delta(S0, K, T, r, sigma, 'call')
initial_gamma = calculate_gamma(S0, K, T, r, sigma)

total_shares = n_contracts * contract_size
total_premium = premium_received * total_shares

print(f"\nTheoretical Option Value: ${initial_option_value:.4f} per share")
print(f"Total Premium Received: ${total_premium:,.2f}")
print(f"Initial Edge: ${(premium_received - initial_option_value) * total_shares:,.2f}")

print(f"\nInitial Greeks:")
print(f"  Delta per option: {initial_delta:.6f}")
print(f"  Total Delta exposure: {initial_delta * total_shares:,.2f} shares")
print(f"  Gamma per option: {initial_gamma:.6f}")

# Initial hedge
shares_to_short = initial_delta * total_shares
hedge_value = shares_to_short * S0

print(f"\nInitial Hedge:")
print(f"  Short {shares_to_short:,.0f} shares at ${S0:.2f}")
print(f"  Hedge Value: ${hedge_value:,.2f}")

# Scenario Analysis: Stock moves to different prices
stock_scenarios = [90, 95, 100, 105, 110]

print(f"\n{'Stock Price':<15} {'New Delta':<12} {'Shares Needed':<18} {'Rebalance':<15} {'PnL':<15}")
print("-" * 80)

for S_new in stock_scenarios:
    # Calculate new delta
    T_remaining = T - 0.04  # 2 weeks passed (~10 trading days)
    new_delta = calculate_delta(S_new, K, T_remaining, r, sigma, 'call')
    new_option_value = black_scholes_price(S_new, K, T_remaining, r, sigma, 'call')
    
    # Required shares for hedge
    shares_needed = new_delta * total_shares
    rebalance = shares_needed - shares_to_short
    
    # Calculate P&L components
    option_pnl = (initial_option_value - new_option_value) * total_shares  # Short options
    hedge_pnl = (S0 - S_new) * shares_to_short  # Short shares
    total_pnl = option_pnl + hedge_pnl
    
    print(f"${S_new:<14.2f} {new_delta:>11.6f} {shares_needed:>17,.0f} {rebalance:>14,.0f} ${total_pnl:>13,.2f}")

print(f"\nKey Insights:")
print(f"  • Market maker profits from selling at premium > theoretical value")
print(f"  • Delta hedging isolates the edge while eliminating directional risk")
print(f"  • Periodic rebalancing needed as delta changes")
print(f"  • P&L driven by gamma (curvature) and theta (time decay)")

# Calculate break-even volatility
realized_vol = sigma * 1.10  # 10% higher than implied
print(f"\nRisk Analysis:")
print(f"  Implied Volatility (sold): {sigma:.1%}")
print(f"  If Realized Volatility: {realized_vol:.1%}")
print(f"  • Higher realized vol → losses from rehedging costs")
print(f"  • Lower realized vol → profits from collecting premium")

print("\n" + "=" * 80)
print('[OK] Market maker delta hedging example complete')

## 4. Vega: Volatility Sensitivity

### Theory

**Vega (ν)** measures the sensitivity of an option's price to changes in implied volatility:

$$
\nu = \frac{\partial V}{\partial \sigma}
$$

where $V$ is the option value and $\sigma$ is the volatility.

### Key Properties of Vega

**Sign and Magnitude**:
- **Long options** (calls and puts): Vega > 0 (benefit from volatility increase)
- **Short options**: Vega < 0 (hurt by volatility increase)
- **ATM options**: Highest vega (maximum sensitivity to volatility changes)
- **Deep ITM/OTM options**: Low vega (minimal volatility sensitivity)

**Vega is Identical for Calls and Puts**: Under Black-Scholes, calls and puts with the same strike and expiration have identical vega.

### Why Vega Matters

1. **Volatility Trading**: Vega measures exposure to volatility changes, key for vol traders
2. **Implied vs Realized Vol**: Profit from mismatch between implied and realized volatility
3. **VIX Trading**: Understanding vega is crucial for trading volatility indices
4. **Risk Management**: Portfolios can have significant vega exposure requiring hedging
5. **Earnings Events**: Options around earnings have elevated vega due to uncertainty

### Vega and Time to Expiration

Vega increases with time to expiration:
- **Long-dated options**: High vega (more time for volatility to matter)
- **Near-expiration options**: Low vega (less time for volatility impact)
- At expiration: Vega → 0

### Volatility Surface

In practice, implied volatility varies by strike and expiration (volatility smile/smirk):
- Market makers manage vega exposure across the volatility surface
- Risk measured by vega exposure to different parts of the surface
- **Vega notional**: Vega weighted by position size

### Mathematical Formulation

For European options under the Black-Scholes model, vega has a closed-form expression:

$$
\nu = S \sqrt{T} \cdot n(d_1)
$$

where:
- $S$ is the stock price
- $T$ is the time to expiration
- $n(d_1)$ is the standard normal probability density function evaluated at $d_1$
- $d_1 = \frac{\ln(S/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}$

The standard normal PDF is:
$$
n(x) = \frac{1}{\sqrt{2\pi}} e^{-x^2/2}
$$

### Key Relationships

**Vega and Gamma Relationship**:

Vega and gamma are related through:
$$
\nu = \sigma S^2 T \cdot \Gamma
$$

This shows that high-gamma positions also have high vega exposure.

**Vega Units**:

Vega is typically quoted as:
- **Dollar change** per 1% change in volatility
- Example: Vega = 0.35 means option price changes by $0.35 for 1% volatility change
- If σ increases from 20% to 21%, option price increases by $0.35

**Numerical Approximation**:

Vega can be approximated using finite differences:
$$
\nu \approx \frac{V(\sigma + \Delta\sigma) - V(\sigma - \Delta\sigma)}{2\Delta\sigma}
$$

Typically use $\Delta\sigma = 0.01$ (1% change in volatility).

**Vega as a Function of Moneyness**:

Vega is maximized for ATM options:
- At $S = K$: Vega is highest
- As $|S - K|$ increases: Vega decreases
- Deep ITM/OTM: Vega → 0 (price determined primarily by intrinsic value)

In [ ]:
# Vega Implementation and Calculation

def calculate_vega(S, K, T, r, sigma):
    """
    Calculate option Vega using Black-Scholes formula.
    
    Vega is the same for both calls and puts.
    
    Parameters
    ----------
    S : float
        Current stock price
    K : float
        Strike price
    T : float
        Time to expiration (years)
    r : float
        Risk-free rate
    sigma : float
        Volatility (annualized)
    
    Returns
    -------
    float
        Option Vega (price change per 1% change in volatility)
    """
    if T <= 0:
        return 0.0
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    vega = S * np.sqrt(T) * norm.pdf(d1) / 100  # Divided by 100 for 1% change
    
    return vega


def calculate_vega_numerical(S, K, T, r, sigma, option_type='call', dsigma=0.01):
    """
    Calculate option Vega using numerical finite difference.
    
    Parameters
    ----------
    S : float
        Current stock price
    K : float
        Strike price
    T : float
        Time to expiration (years)
    r : float
        Risk-free rate
    sigma : float
        Volatility (annualized)
    option_type : str
        'call' or 'put'
    dsigma : float
        Small change in volatility for finite difference
    
    Returns
    -------
    float
        Option Vega (numerical approximation)
    """
    V_up = black_scholes_price(S, K, T, r, sigma + dsigma, option_type)
    V_down = black_scholes_price(S, K, T, r, sigma - dsigma, option_type)
    
    vega = (V_up - V_down) / (2 * dsigma * 100)  # Per 1% change
    return vega


# Example: Calculate Vega for ATM Option
print("=" * 80)
print("VEGA CALCULATION: VOLATILITY SENSITIVITY")
print("=" * 80)

S = 100
K = 100
T = 0.5  # 6 months
r = 0.05
sigma = 0.25

vega_analytical = calculate_vega(S, K, T, r, sigma)
vega_numerical = calculate_vega_numerical(S, K, T, r, sigma, 'call')
option_price_call = black_scholes_price(S, K, T, r, sigma, 'call')
option_price_put = black_scholes_price(S, K, T, r, sigma, 'put')

print(f"\nOption Parameters:")
print(f"  Stock Price: ${S:.2f}")
print(f"  Strike Price: ${K:.2f} (ATM)")
print(f"  Time to Expiration: {T*12:.1f} months")
print(f"  Risk-Free Rate: {r:.2%}")
print(f"  Implied Volatility: {sigma:.1%}")

print(f"\nOption Prices:")
print(f"  Call Option: ${option_price_call:.4f}")
print(f"  Put Option: ${option_price_put:.4f}")

print(f"\nVega Calculations:")
print(f"  Analytical (Black-Scholes): {vega_analytical:.6f}")
print(f"  Numerical (Finite Difference): {vega_numerical:.6f}")
print(f"  Difference: {abs(vega_analytical - vega_numerical):.8f}")

print(f"\nInterpretation:")
print(f"  • Vega = {vega_analytical:.4f} means:")
print(f"    - Option price changes by ${vega_analytical:.4f} for 1% change in volatility")
print(f"    - If vol increases from {sigma:.1%} to {sigma+0.01:.1%}, price increases by ${vega_analytical:.4f}")
print(f"    - If vol decreases from {sigma:.1%} to {sigma-0.01:.1%}, price decreases by ${vega_analytical:.4f}")

# Demonstrate volatility impact
vol_scenarios = [0.20, 0.25, 0.30, 0.35]
print(f"\nVolatility Impact on Option Price:")
print(f"  {'Volatility':<15} {'Call Price':<15} {'Put Price':<15} {'Price Change':<15}")
print("-" * 70)

base_price = black_scholes_price(S, K, T, r, sigma, 'call')
for vol in vol_scenarios:
    call_price = black_scholes_price(S, K, T, r, vol, 'call')
    put_price = black_scholes_price(S, K, T, r, vol, 'put')
    price_change = call_price - base_price
    print(f"  {vol:.1%:<14} ${call_price:>13.4f} ${put_price:>13.4f} ${price_change:>13.4f}")

# Calculate vega for portfolio of 100 contracts
n_contracts = 100
contract_size = 100
total_vega = vega_analytical * n_contracts * contract_size

print(f"\nPortfolio Vega (Long {n_contracts} Contracts):")
print(f"  Vega per option: {vega_analytical:.6f}")
print(f"  Total contracts: {n_contracts}")
print(f"  Shares per contract: {contract_size}")
print(f"  Total Portfolio Vega: ${total_vega:,.2f}")
print(f"\n  Interpretation:")
print(f"    - For 1% increase in volatility: Portfolio gains ${total_vega:,.2f}")
print(f"    - For 5% increase in volatility: Portfolio gains ${total_vega * 5:,.2f}")
print(f"    - Positive vega: Benefits from volatility increase")

print("\n" + "=" * 80)
print('[OK] Vega calculation complete')

In [ ]:
# Visualization: Vega Across Strike, Time, and Volatility

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Vega vs Stock Price
S_range = np.linspace(70, 130, 100)
K = 100
T = 0.5
r = 0.05
sigma = 0.25

vegas = [calculate_vega(S, K, T, r, sigma) for S in S_range]

ax1.plot(S_range, vegas, 'purple', linewidth=2.5)
ax1.axvline(x=K, color='red', linestyle='--', linewidth=1.5, label=f'Strike K=${K}')
ax1.fill_between(S_range, 0, vegas, alpha=0.3, color='purple')

# Mark maximum
max_vega_idx = np.argmax(vegas)
ax1.plot(S_range[max_vega_idx], vegas[max_vega_idx], 'ro', markersize=10, label='Maximum Vega')
ax1.annotate(f'Max Vega: {vegas[max_vega_idx]:.4f}', 
            xy=(S_range[max_vega_idx], vegas[max_vega_idx]),
            xytext=(S_range[max_vega_idx]+5, vegas[max_vega_idx]+0.01),
            fontsize=10, bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

ax1.set_xlabel('Stock Price ($)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Vega ($ per 1% vol change)', fontsize=12, fontweight='bold')
ax1.set_title('Vega vs Stock Price (T=6mo, σ=25%)', fontsize=13, fontweight='bold')
ax1.legend(loc='best', fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 2: Vega vs Time to Expiration
T_range = np.linspace(0.05, 2.0, 100)
S = 100
K = 100

vegas_time = [calculate_vega(S, K, T_val, r, sigma) for T_val in T_range]

ax2.plot(T_range, vegas_time, 'darkgreen', linewidth=2.5)
ax2.fill_between(T_range, 0, vegas_time, alpha=0.3, color='green')

ax2.set_xlabel('Time to Expiration (years)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Vega ($ per 1% vol change)', fontsize=12, fontweight='bold')
ax2.set_title('ATM Vega vs Time to Expiration (S=K=$100)', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Add annotation
ax2.text(0.5, 0.95, 'Vega increases with time:\nMore time = more volatility exposure',
        transform=ax2.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

# Plot 3: Vega vs Strike for Different Expirations
K_range = np.linspace(70, 130, 100)
S = 100
T_scenarios = [0.25, 0.5, 1.0]

for T_val in T_scenarios:
    vegas_strike = [calculate_vega(S, K_val, T_val, r, sigma) for K_val in K_range]
    ax3.plot(K_range, vegas_strike, linewidth=2, label=f'T={T_val:.2f}y ({T_val*12:.0f}mo)')

ax3.axvline(x=S, color='red', linestyle='--', linewidth=1.5, label=f'Stock S=${S}')
ax3.set_xlabel('Strike Price ($)', fontsize=12, fontweight='bold')
ax3.set_ylabel('Vega ($ per 1% vol change)', fontsize=12, fontweight='bold')
ax3.set_title('Vega vs Strike for Different Expirations', fontsize=13, fontweight='bold')
ax3.legend(loc='best', fontsize=10)
ax3.grid(True, alpha=0.3)

# Plot 4: Vega vs Volatility Level
sigma_range = np.linspace(0.10, 0.60, 100)
S = 100
K = 100
T = 0.5

vegas_vol = [calculate_vega(S, K, T, r, sig) for sig in sigma_range]

ax4.plot(sigma_range * 100, vegas_vol, 'darkorange', linewidth=2.5)
ax4.axvline(x=sigma*100, color='blue', linestyle='--', linewidth=1.5, 
           label=f'Current σ={sigma:.1%}')

ax4.set_xlabel('Volatility (%)', fontsize=12, fontweight='bold')
ax4.set_ylabel('Vega ($ per 1% vol change)', fontsize=12, fontweight='bold')
ax4.set_title('ATM Vega vs Volatility Level (S=K=$100, T=6mo)', fontsize=13, fontweight='bold')
ax4.legend(loc='best', fontsize=10)
ax4.grid(True, alpha=0.3)

# Add annotation showing vega decreases with higher vol
ax4.text(0.7, 0.95, 'Vega decreases at\nhigher volatility levels',
        transform=ax4.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))

plt.tight_layout()
plt.show()

print('[OK] Vega visualization complete')

### Practical Example: Volatility Trading Strategy

Let's analyze a straddle strategy to understand vega exposure and how traders profit from volatility.

In [ ]:
# Practical Example: Long Straddle Vega Exposure

print("=" * 80)
print("VOLATILITY TRADING: LONG STRADDLE STRATEGY")
print("=" * 80)

# Strategy: Buy ATM call and ATM put (long straddle)
S = 100
K = 100  # ATM straddle
T = 0.25  # 3 months to earnings announcement
r = 0.05
sigma_current = 0.30  # Current implied volatility

print(f"\nStrategy: Long ATM Straddle")
print(f"  Expectation: Volatility will increase (e.g., before earnings)")
print(f"\nMarket Conditions:")
print(f"  Stock Price: ${S:.2f}")
print(f"  Strike: ${K:.2f} (ATM)")
print(f"  Time to Earnings: {T*252:.0f} trading days")
print(f"  Current Implied Vol: {sigma_current:.1%}")

# Calculate initial position
call_price = black_scholes_price(S, K, T, r, sigma_current, 'call')
put_price = black_scholes_price(S, K, T, r, sigma_current, 'put')
straddle_cost = call_price + put_price

call_delta = calculate_delta(S, K, T, r, sigma_current, 'call')
put_delta = calculate_delta(S, K, T, r, sigma_current, 'put')
straddle_delta = call_delta + put_delta

vega = calculate_vega(S, K, T, r, sigma_current)
straddle_vega = 2 * vega  # Both call and put have same vega

print(f"\nInitial Position (per share):")
print(f"  Call Price: ${call_price:.4f}")
print(f"  Put Price: ${put_price:.4f}")
print(f"  Straddle Cost: ${straddle_cost:.4f}")

print(f"\nGreeks:")
print(f"  Call Delta: {call_delta:.4f}")
print(f"  Put Delta: {put_delta:.4f}")
print(f"  Straddle Delta: {straddle_delta:.4f} (approximately zero)")
print(f"  Vega per option: {vega:.4f}")
print(f"  Straddle Vega: {straddle_vega:.4f}")

# Scenario 1: Volatility increases (earnings anticipation)
sigma_scenarios = [0.30, 0.35, 0.40, 0.45, 0.50]

print(f"\nScenario Analysis: Impact of Volatility Changes (Stock stays at ${S})")
print(f"  {'Volatility':<15} {'Straddle Value':<18} {'P&L':<15} {'% Return':<15}")
print("-" * 70)

for sig in sigma_scenarios:
    call_value = black_scholes_price(S, K, T, r, sig, 'call')
    put_value = black_scholes_price(S, K, T, r, sig, 'put')
    straddle_value = call_value + put_value
    pnl = straddle_value - straddle_cost
    pct_return = (pnl / straddle_cost) * 100
    
    print(f"  {sig:.1%:<14} ${straddle_value:>16.4f} ${pnl:>13.4f} {pct_return:>13.2f}%")

# Portfolio Analysis: 100 contracts
n_contracts = 100
contract_size = 100
total_vega = straddle_vega * n_contracts * contract_size
total_cost = straddle_cost * n_contracts * contract_size

print(f"\nPortfolio Analysis ({n_contracts} Straddle Contracts):")
print(f"  Total Investment: ${total_cost:,.2f}")
print(f"  Portfolio Vega: ${total_vega:,.2f}")
print(f"\n  P&L for Volatility Changes:")

for vol_change in [1, 5, 10]:
    pnl_estimate = total_vega * vol_change
    pct_return = (pnl_estimate / total_cost) * 100
    print(f"    +{vol_change}% vol increase: ${pnl_estimate:,.2f} ({pct_return:,.2f}% return)")

# Risk Analysis
print(f"\nRisk Considerations:")
print(f"  • Positive vega: Profits if volatility increases")
print(f"  • Near-zero delta: Minimal directional risk")
print(f"  • Theta exposure: Time decay works against position")
print(f"  • Break-even: Stock must move >${straddle_cost:.2f} or volatility must rise")

# Scenario 2: After earnings, volatility collapses
T_after = T - (10/252)  # 10 days after earnings
sigma_post = 0.25  # Volatility drops after event

call_post = black_scholes_price(S, K, T_after, r, sigma_post, 'call')
put_post = black_scholes_price(S, K, T_after, r, sigma_post, 'put')
straddle_post = call_post + put_post
pnl_post = straddle_post - straddle_cost

print(f"\nPost-Earnings Scenario (Volatility Crush):")
print(f"  Time passed: 10 trading days")
print(f"  New volatility: {sigma_post:.1%} (crush from {sigma_current:.1%})")
print(f"  Stock price unchanged: ${S:.2f}")
print(f"  Straddle Value: ${straddle_post:.4f}")
print(f"  P&L: ${pnl_post:.4f} ({(pnl_post/straddle_cost)*100:.2f}%)")
print(f"  Result: Loss from vol crush despite being right on direction!")

print("\n" + "=" * 80)
print('[OK] Volatility trading example complete')

## 5. Theta: Time Decay

### Theory

**Theta (Θ)** measures the rate of change of an option's value with respect to the passage of time:

$$
\Theta = \frac{\partial V}{\partial t}
$$

Theta represents **time decay** - the erosion of option value as expiration approaches, holding all else constant.

### Key Properties of Theta

**Sign Convention**:
- **Long options**: Theta < 0 (lose value over time)
- **Short options**: Theta > 0 (gain value over time)
- Theta is typically expressed as the dollar loss per day

**Magnitude by Moneyness**:
- **ATM options**: Highest absolute theta (fastest decay)
- **Deep ITM/OTM options**: Lower absolute theta (slower decay)
- At expiration: All time value → 0

### Why Theta Matters

1. **Time is Money**: Theta quantifies the cost of holding long option positions
2. **Premium Collection**: Short option strategies profit from theta decay
3. **Strategy Selection**: Balance theta against other Greeks based on market view
4. **Expiration Selection**: Near-term options have higher theta than long-term
5. **Risk Management**: Theta exposure can offset gamma/vega risk

### Theta and Option Lifecycle

**Time Decay Acceleration**:
- Theta increases (in absolute value) as expiration approaches
- Final 30 days: Rapid acceleration of time decay
- Weekend and holiday effects: Theta accrues even when market is closed

### Theta-Gamma Relationship

Theta and gamma are inversely related:
- High gamma positions → High negative theta
- This relationship comes from the Black-Scholes PDE
- Delta-hedged portfolios: Theta P&L offsets gamma P&L

### Mathematical Formulation

For European options under Black-Scholes, theta has closed-form expressions:

**Call Option Theta**:
$$
\Theta_{\text{call}} = -\frac{S n(d_1) \sigma}{2\sqrt{T}} - rKe^{-rT}N(d_2)
$$

**Put Option Theta**:
$$
\Theta_{\text{put}} = -\frac{S n(d_1) \sigma}{2\sqrt{T}} + rKe^{-rT}N(-d_2)
$$

where:
- $n(d_1)$ is the standard normal PDF evaluated at $d_1$
- $N(\cdot)$ is the standard normal CDF
- $d_1, d_2$ are the Black-Scholes d parameters

**Conversion to Daily Theta**:

The formulas above give theta in years. For daily theta:
$$
\Theta_{\text{daily}} = \frac{\Theta_{\text{annual}}}{365}
$$

Some practitioners use 252 (trading days) instead of 365.

### Black-Scholes PDE Relationship

From the Black-Scholes PDE, for a delta-hedged portfolio:
$$
\Theta + \frac{1}{2}\sigma^2 S^2 \Gamma + rS\Delta - rV = 0
$$

For a delta-neutral portfolio ($\Delta = 0$):
$$
\Theta + \frac{1}{2}\sigma^2 S^2 \Gamma = rV
$$

This shows the fundamental **theta-gamma trade-off**:
- Negative theta (time decay) is compensated by positive gamma (convexity)
- The compensation rate is the risk-free rate

### Theta and Time Value

Theta represents the decay of time value only:
$$
V = \text{Intrinsic Value} + \text{Time Value}
$$

Since intrinsic value doesn't decay:
$$
\Theta = -\frac{\partial (\text{Time Value})}{\partial t}
$$

In [ ]:
# Theta Implementation and Calculation

def calculate_theta(S, K, T, r, sigma, option_type='call', days_per_year=365):
    """
    Calculate option Theta using Black-Scholes formula.
    
    Parameters
    ----------
    S : float
        Current stock price
    K : float
        Strike price
    T : float
        Time to expiration (years)
    r : float
        Risk-free rate
    sigma : float
        Volatility (annualized)
    option_type : str
        'call' or 'put'
    days_per_year : int
        Convention for daily theta (365 or 252)
    
    Returns
    -------
    float
        Option Theta (price change per day)
    """
    if T <= 0:
        return 0.0
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    # Common term for both calls and puts
    term1 = -(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))
    
    if option_type == 'call':
        term2 = -r * K * np.exp(-r * T) * norm.cdf(d2)
        theta = (term1 + term2) / days_per_year
    else:
        term2 = r * K * np.exp(-r * T) * norm.cdf(-d2)
        theta = (term1 + term2) / days_per_year
    
    return theta


def calculate_theta_numerical(S, K, T, r, sigma, option_type='call', dt=1/365):
    """
    Calculate option Theta using numerical finite difference.
    
    Parameters
    ----------
    S : float
        Current stock price
    K : float
        Strike price
    T : float
        Time to expiration (years)
    r : float
        Risk-free rate
    sigma : float
        Volatility (annualized)
    option_type : str
        'call' or 'put'
    dt : float
        Time step for finite difference (default: 1 day)
    
    Returns
    -------
    float
        Option Theta (numerical approximation)
    """
    if T <= dt:
        return 0.0
    
    V_now = black_scholes_price(S, K, T, r, sigma, option_type)
    V_next = black_scholes_price(S, K, T - dt, r, sigma, option_type)
    
    theta = V_next - V_now  # Already per dt period
    return theta


# Example: Calculate Theta for ATM Option
print("=" * 80)
print("THETA CALCULATION: TIME DECAY")
print("=" * 80)

S = 100
K = 100
T = 0.25  # 3 months
r = 0.05
sigma = 0.25

theta_call = calculate_theta(S, K, T, r, sigma, 'call')
theta_put = calculate_theta(S, K, T, r, sigma, 'put')
theta_call_num = calculate_theta_numerical(S, K, T, r, sigma, 'call')

option_price_call = black_scholes_price(S, K, T, r, sigma, 'call')
option_price_put = black_scholes_price(S, K, T, r, sigma, 'put')

gamma = calculate_gamma(S, K, T, r, sigma)

print(f"\nOption Parameters:")
print(f"  Stock Price: ${S:.2f}")
print(f"  Strike: ${K:.2f} (ATM)")
print(f"  Time to Expiration: {T*365:.0f} days")
print(f"  Volatility: {sigma:.1%}")

print(f"\nOption Prices:")
print(f"  Call: ${option_price_call:.4f}")
print(f"  Put: ${option_price_put:.4f}")

print(f"\nTheta Calculations (per day):")
print(f"  Call Theta (analytical): ${theta_call:.6f}")
print(f"  Call Theta (numerical): ${theta_call_num:.6f}")
print(f"  Put Theta: ${theta_put:.6f}")

print(f"\nInterpretation:")
print(f"  • Call loses ${abs(theta_call):.4f} per day from time decay")
print(f"  • Put loses ${abs(theta_put):.4f} per day from time decay")
print(f"  • Over 1 week: Call loses ${abs(theta_call)*7:.4f}")
print(f"  • Over 30 days: Call loses ${abs(theta_call)*30:.4f}")

# Time decay over the option's life
print(f"\nTime Decay Analysis:")
days_remaining = [63, 45, 30, 15, 5, 1]

print(f"  {'Days to Expiry':<18} {'Call Price':<15} {'Daily Theta':<18} {'% of Value':<15}")
print("-" * 75)

for days in days_remaining:
    T_remaining = days / 365
    if T_remaining > 0:
        call_price = black_scholes_price(S, K, T_remaining, r, sigma, 'call')
        theta_day = calculate_theta(S, K, T_remaining, r, sigma, 'call')
        theta_pct = (abs(theta_day) / call_price) * 100
        print(f"  {days:<18} ${call_price:>13.4f} ${theta_day:>16.6f} {theta_pct:>13.2f}%")

# Theta-Gamma relationship
print(f"\nTheta-Gamma Relationship (Delta-Hedged Portfolio):")
print(f"  Gamma: {gamma:.6f}")
print(f"  Theta (Call): ${theta_call:.6f} per day")
print(f"\n  From Black-Scholes PDE:")
theta_theoretical = -(0.5 * sigma**2 * S**2 * gamma) / 365
print(f"  Theoretical Theta: ${theta_theoretical:.6f} per day")
print(f"  (Approximation ignoring interest rate terms)")

print(f"\n  Key Insight:")
print(f"    • Negative theta (time decay) is compensated by positive gamma")
print(f"    • Delta-hedged portfolio: Theta loss offset by gamma gains from rehedging")

# Portfolio analysis
n_contracts = 100
contract_size = 100
total_theta = theta_call * n_contracts * contract_size

print(f"\nPortfolio Theta (Long {n_contracts} Call Contracts):")
print(f"  Daily Theta Loss: ${abs(total_theta):,.2f}")
print(f"  Weekly Theta Loss: ${abs(total_theta)*7:,.2f}")
print(f"  Monthly Theta Loss: ${abs(total_theta)*30:,.2f}")

print("\n" + "=" * 80)
print('[OK] Theta calculation complete')

In [ ]:
# Visualization: Theta Decay Patterns

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# Parameters
S = 100
K = 100
r = 0.05
sigma = 0.25

# Plot 1: Theta vs Stock Price
S_range = np.linspace(70, 130, 100)
T = 0.25

thetas_call = [calculate_theta(S_val, K, T, r, sigma, 'call') for S_val in S_range]
thetas_put = [calculate_theta(S_val, K, T, r, sigma, 'put') for S_val in S_range]

ax1.plot(S_range, thetas_call, 'b-', linewidth=2, label='Call Theta')
ax1.plot(S_range, thetas_put, 'r-', linewidth=2, label='Put Theta')
ax1.axvline(x=K, color='green', linestyle='--', linewidth=1.5, label=f'Strike K=${K}')
ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

ax1.set_xlabel('Stock Price ($)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Theta ($ per day)', fontsize=12, fontweight='bold')
ax1.set_title('Theta vs Stock Price (T=3mo)', fontsize=13, fontweight='bold')
ax1.legend(loc='best', fontsize=10)
ax1.grid(True, alpha=0.3)

# Annotate ATM region
ax1.annotate('ATM: Maximum\n|Theta|', xy=(K, thetas_call[50]), 
            xytext=(K+10, thetas_call[50]-0.01),
            fontsize=9, bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7),
            arrowprops=dict(arrowstyle='->', color='black'))

# Plot 2: Theta vs Time to Expiration (Time Decay Curve)
T_range = np.linspace(0.02, 1.0, 100)
S = 100

thetas_time = [calculate_theta(S, K, T_val, r, sigma, 'call') for T_val in T_range]
option_prices = [black_scholes_price(S, K, T_val, r, sigma, 'call') for T_val in T_range]

ax2.plot(T_range * 365, thetas_time, 'darkred', linewidth=2.5)
ax2.fill_between(T_range * 365, 0, thetas_time, alpha=0.3, color='red')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

ax2.set_xlabel('Days to Expiration', fontsize=12, fontweight='bold')
ax2.set_ylabel('Theta ($ per day)', fontsize=12, fontweight='bold')
ax2.set_title('ATM Call Theta vs Time to Expiration', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Add annotation showing acceleration
ax2.text(0.6, 0.05, 'Theta accelerates\nas expiration approaches',
        transform=ax2.transAxes, fontsize=10,
        bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.7))

# Plot 3: Time Decay Simulation (Option Value Over Time)
days_to_expiry = np.linspace(90, 0, 91)
T_vals = days_to_expiry / 365

# Multiple moneyness scenarios
scenarios = {'ITM (S=$110)': 110, 'ATM (S=$100)': 100, 'OTM (S=$90)': 90}
colors = {'ITM (S=$110)': 'green', 'ATM (S=$100)': 'blue', 'OTM (S=$90)': 'red'}

for label, S_scenario in scenarios.items():
    prices = [black_scholes_price(S_scenario, K, T_val, r, sigma, 'call') if T_val > 0 
              else max(S_scenario - K, 0) for T_val in T_vals]
    ax3.plot(days_to_expiry, prices, linewidth=2.5, label=label, color=colors[label])
    
    # Mark intrinsic value
    intrinsic = max(S_scenario - K, 0)
    ax3.axhline(y=intrinsic, color=colors[label], linestyle=':', linewidth=1, alpha=0.5)

ax3.set_xlabel('Days to Expiration', fontsize=12, fontweight='bold')
ax3.set_ylabel('Option Price ($)', fontsize=12, fontweight='bold')
ax3.set_title('Time Decay: Option Value Over Time (K=$100)', fontsize=13, fontweight='bold')
ax3.legend(loc='best', fontsize=10)
ax3.grid(True, alpha=0.3)
ax3.invert_xaxis()  # Time moves forward (right to left)

# Add shaded region for final 30 days
ax3.axvspan(0, 30, alpha=0.2, color='yellow', label='Rapid Decay Zone')

# Plot 4: Theta-Gamma Relationship
S_range_tg = np.linspace(80, 120, 100)
T = 0.25

thetas = np.array([calculate_theta(S_val, K, T, r, sigma, 'call') for S_val in S_range_tg])
gammas = np.array([calculate_gamma(S_val, K, T, r, sigma) for S_val in S_range_tg])

# Normalize for visualization
theta_normalized = abs(thetas) / np.max(abs(thetas))
gamma_normalized = gammas / np.max(gammas)

ax4_twin = ax4.twinx()

line1 = ax4.plot(S_range_tg, abs(thetas), 'r-', linewidth=2.5, label='|Theta|')
line2 = ax4_twin.plot(S_range_tg, gammas, 'b-', linewidth=2.5, label='Gamma')

ax4.axvline(x=K, color='green', linestyle='--', linewidth=1.5, alpha=0.7, label='Strike')

ax4.set_xlabel('Stock Price ($)', fontsize=12, fontweight='bold')
ax4.set_ylabel('|Theta| ($ per day)', fontsize=12, fontweight='bold', color='red')
ax4_twin.set_ylabel('Gamma', fontsize=12, fontweight='bold', color='blue')
ax4.set_title('Theta-Gamma Relationship (T=3mo)', fontsize=13, fontweight='bold')

# Combine legends
lines1, labels1 = ax4.get_legend_handles_labels()
lines2, labels2 = ax4_twin.get_legend_handles_labels()
ax4.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=10)

ax4.tick_params(axis='y', labelcolor='red')
ax4_twin.tick_params(axis='y', labelcolor='blue')
ax4.grid(True, alpha=0.3)

# Add annotation
ax4.text(0.05, 0.95, 'High Gamma\n↓\nHigh |Theta|',
        transform=ax4.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

plt.tight_layout()
plt.show()

print('[OK] Theta visualization complete')

### Practical Example: Covered Call Theta Strategy

Let's analyze how theta decay benefits covered call writers who collect premium.

In [ ]:
# Practical Example: Covered Call Strategy with Theta Decay

print("=" * 80)
print("THETA STRATEGY: COVERED CALL (SHORT OTM CALL)")
print("=" * 80)

# Strategy: Own 1000 shares, sell 10 OTM call contracts
shares_owned = 1000
n_contracts = 10  # Each contract = 100 shares
contract_size = 100

S = 100
K = 105  # 5% OTM
T = 45/365  # 45 days to expiration
r = 0.05
sigma = 0.25

print(f"\nCovered Call Strategy:")
print(f"  • Long {shares_owned} shares at ${S:.2f}")
print(f"  • Short {n_contracts} call contracts (Strike ${K:.2f})")
print(f"  • Days to expiration: {T*365:.0f}")
print(f"  • Implied volatility: {sigma:.1%}")

# Calculate option parameters
call_price = black_scholes_price(S, K, T, r, sigma, 'call')
call_delta = calculate_delta(S, K, T, r, sigma, 'call')
call_theta = calculate_theta(S, K, T, r, sigma, 'call')
call_vega = calculate_vega(S, K, T, r, sigma)

premium_collected = call_price * n_contracts * contract_size

print(f"\nOption Sold:")
print(f"  Call Price: ${call_price:.4f} per share")
print(f"  Premium Collected: ${premium_collected:,.2f}")
print(f"  Yield on Stock: {(premium_collected / (shares_owned * S)) * 100:.2f}% for {T*365:.0f} days")
print(f"  Annualized Yield: {(premium_collected / (shares_owned * S)) * (365 / (T*365)) * 100:.2f}%")

print(f"\nGreeks (per option):")
print(f"  Delta: {call_delta:.4f}")
print(f"  Theta: ${call_theta:.6f} per day")
print(f"  Vega: {call_vega:.4f}")

# Portfolio Greeks (short position)
portfolio_delta = shares_owned - (call_delta * n_contracts * contract_size)
portfolio_theta = -(call_theta * n_contracts * contract_size)  # Short position: positive theta

print(f"\nPortfolio Greeks (Covered Call):")
print(f"  Stock Delta: {shares_owned:,.0f}")
print(f"  Short Calls Delta: {-(call_delta * n_contracts * contract_size):,.2f}")
print(f"  Net Portfolio Delta: {portfolio_delta:,.2f}")
print(f"  Portfolio Theta: ${portfolio_theta:,.2f} per day (POSITIVE - we collect decay!)")

# Simulate theta decay over time
days_sequence = np.arange(0, int(T*365) + 1)
option_values = []
theta_collected = []

print(f"\nTheta Decay Simulation (if stock stays at ${S}):")
print(f"  {'Day':<8} {'Call Value':<15} {'Daily Theta':<18} {'Cumulative Theta':<20}")
print("-" * 75)

cumulative_theta = 0
for day in [0, 10, 20, 30, 40, 45]:
    if day < T*365:
        T_remaining = (T*365 - day) / 365
        call_value = black_scholes_price(S, K, T_remaining, r, sigma, 'call')
        theta_day = calculate_theta(S, K, T_remaining, r, sigma, 'call')
        
        # Theta collected by short position
        if day > 0:
            cumulative_theta += abs(theta_day) * n_contracts * contract_size
        
        print(f"  {day:<8} ${call_value:>13.4f} ${abs(theta_day):>16.6f} ${cumulative_theta:>18.2f}")

# Scenario analysis at expiration
print(f"\nExpiration Scenarios:")
stock_prices_exp = [95, 100, 105, 110, 115]

print(f"  {'Stock Price':<15} {'Call Value':<15} {'Stock P&L':<15} {'Option P&L':<15} {'Total P&L':<15}")
print("-" * 85)

for S_exp in stock_prices_exp:
    # Call value at expiration
    call_value_exp = max(S_exp - K, 0)
    
    # P&L components
    stock_pnl = (S_exp - S) * shares_owned
    option_pnl = (call_price - call_value_exp) * n_contracts * contract_size
    total_pnl = stock_pnl + option_pnl
    
    print(f"  ${S_exp:<14.0f} ${call_value_exp:>13.4f} ${stock_pnl:>13.2f} ${option_pnl:>13.2f} ${total_pnl:>13.2f}")

# Risk analysis
print(f"\nRisk/Return Profile:")
print(f"  Maximum Profit: ${(K - S) * shares_owned + premium_collected:,.2f}")
print(f"    (Stock rises to ${K}, calls expire worthless)")
print(f"  Breakeven: ${S - (premium_collected / shares_owned):.2f}")
print(f"    (Stock falls, but premium cushions loss)")
print(f"  Maximum Loss: Unlimited downside on stock (partially offset by premium)")

# Theta benefit calculation
days_held = 30
T_after_30 = T - (days_held/365)
call_value_after = black_scholes_price(S, K, T_after_30, r, sigma, 'call')
theta_profit = (call_price - call_value_after) * n_contracts * contract_size

print(f"\nTheta Benefit (holding {days_held} days, stock unchanged):")
print(f"  Initial Call Value: ${call_price:.4f}")
print(f"  Value after {days_held} days: ${call_value_after:.4f}")
print(f"  Decay per option: ${call_price - call_value_after:.4f}")
print(f"  Total Theta Profit: ${theta_profit:,.2f}")
print(f"  Return on Stock: {(theta_profit / (shares_owned * S)) * 100:.2f}% in {days_held} days")

print(f"\nKey Insights:")
print(f"  • Covered call converts stock position into positive theta strategy")
print(f"  • Collect premium as time passes, even if stock doesn't move")
print(f"  • Trade-off: Cap upside potential at strike price")
print(f"  • Ideal for neutral-to-bullish outlook with income generation")

print("\n" + "=" * 80)
print('[OK] Covered call theta strategy complete')

## 6. Rho: Interest Rate Sensitivity

### Theory

**Rho (ρ)** measures the sensitivity of an option's price to changes in the risk-free interest rate:

$$
\rho = \frac{\partial V}{\partial r}
$$

where $V$ is the option value and $r$ is the risk-free rate.

### Key Properties of Rho

**Sign by Option Type**:
- **Call options**: Rho > 0 (benefit from rate increases)
- **Put options**: Rho < 0 (hurt by rate increases)
- Intuition: Higher rates increase call values through present value of strike payment

**Magnitude Characteristics**:
- **Long-dated options**: Higher rho (more time for rates to matter)
- **Short-dated options**: Low rho (minimal rate impact)
- **Deep ITM options**: Higher absolute rho
- **OTM options**: Lower absolute rho

### Why Rho Matters (Sometimes)

1. **Low Importance in Normal Markets**: Rho is typically the least important Greek
2. **Relevant for Long-Dated Options**: LEAPS and multi-year options have meaningful rho
3. **Rate Volatility Periods**: When central banks actively change rates (2022-2023)
4. **Fixed Income Hybrids**: Convertible bonds, equity-linked notes
5. **Cross-Currency Options**: FX options have significant rate differential exposure

### Rho in Different Rate Environments

**Rising Rate Environment** (2022-2024):
- Call options gain value (positive rho)
- Put options lose value (negative rho)
- Short-term options: minimal impact
- Long-term options: noticeable impact

**Low Rate Environment** (2009-2021):
- Rho largely ignored by traders
- Minimal impact across all options
- Other Greeks dominate P&L

### Mathematical Formulation

For European options under Black-Scholes, rho has closed-form expressions:

**Call Option Rho**:
$$
\rho_{\text{call}} = K T e^{-rT} N(d_2)
$$

**Put Option Rho**:
$$
\rho_{\text{put}} = -K T e^{-rT} N(-d_2)
$$

where:
- $K$ is the strike price
- $T$ is time to expiration
- $N(\cdot)$ is the standard normal CDF
- $d_2 = d_1 - \sigma\sqrt{T}$

**Rho Units**:

Rho is typically quoted as:
- **Dollar change** per 1% (100 basis points) change in interest rate
- Example: Rho = 0.45 means option price changes by $0.45 for 1% rate change
- If $r$ increases from 5% to 6%, option price increases by $0.45

### Intuition Behind Rho

**Why Calls Have Positive Rho**:

The call option value includes the present value of paying the strike:
$$
C = S N(d_1) - Ke^{-rT} N(d_2)
$$

Higher $r$ → Lower $e^{-rT}$ → Lower present value of strike payment → Higher call value

**Why Puts Have Negative Rho**:

The put option value includes receiving the strike:
$$
P = Ke^{-rT} N(-d_2) - S N(-d_1)
$$

Higher $r$ → Lower $e^{-rT}$ → Lower present value of strike received → Lower put value

### Numerical Approximation

Rho can be approximated using finite differences:
$$
\rho \approx \frac{V(r + \Delta r) - V(r - \Delta r)}{2\Delta r \times 100}
$$

Typically use $\Delta r = 0.0001$ (1 basis point).

In [ ]:
# Rho Implementation and Calculation

def calculate_rho(S, K, T, r, sigma, option_type='call'):
    """
    Calculate option Rho using Black-Scholes formula.
    
    Parameters
    ----------
    S : float
        Current stock price
    K : float
        Strike price
    T : float
        Time to expiration (years)
    r : float
        Risk-free rate
    sigma : float
        Volatility (annualized)
    option_type : str
        'call' or 'put'
    
    Returns
    -------
    float
        Option Rho (price change per 1% change in interest rate)
    """
    if T <= 0:
        return 0.0
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    if option_type == 'call':
        rho = K * T * np.exp(-r * T) * norm.cdf(d2) / 100  # Per 1% change
    else:
        rho = -K * T * np.exp(-r * T) * norm.cdf(-d2) / 100  # Per 1% change
    
    return rho


def calculate_rho_numerical(S, K, T, r, sigma, option_type='call', dr=0.0001):
    """
    Calculate option Rho using numerical finite difference.
    
    Parameters
    ----------
    S : float
        Current stock price
    K : float
        Strike price
    T : float
        Time to expiration (years)
    r : float
        Risk-free rate
    sigma : float
        Volatility (annualized)
    option_type : str
        'call' or 'put'
    dr : float
        Small change in rate for finite difference (default: 1 bp)
    
    Returns
    -------
    float
        Option Rho (numerical approximation)
    """
    V_up = black_scholes_price(S, K, T, r + dr, sigma, option_type)
    V_down = black_scholes_price(S, K, T, r - dr, sigma, option_type)
    
    rho = (V_up - V_down) / (2 * dr * 100)  # Per 1% change
    return rho


# Example: Calculate Rho for Different Expirations
print("=" * 80)
print("RHO CALCULATION: INTEREST RATE SENSITIVITY")
print("=" * 80)

S = 100
K = 100
r = 0.05
sigma = 0.25

print(f"\nOption Parameters:")
print(f"  Stock Price: ${S:.2f}")
print(f"  Strike: ${K:.2f} (ATM)")
print(f"  Interest Rate: {r:.2%}")
print(f"  Volatility: {sigma:.1%}")

# Compare rho across different expirations
expirations = [
    (30/365, "1 month"),
    (90/365, "3 months"),
    (180/365, "6 months"),
    (1.0, "1 year"),
    (2.0, "2 years (LEAPS)")
]

print(f"\nRho Across Different Expirations:")
print(f"  {'Expiration':<20} {'Call Rho':<15} {'Put Rho':<15} {'Call Price':<15}")
print("-" * 75)

for T, label in expirations:
    rho_call = calculate_rho(S, K, T, r, sigma, 'call')
    rho_put = calculate_rho(S, K, T, r, sigma, 'put')
    call_price = black_scholes_price(S, K, T, r, sigma, 'call')
    
    print(f"  {label:<20} {rho_call:>14.6f} {rho_put:>14.6f} ${call_price:>13.4f}")

# Scenario: Rate changes
T = 1.0  # 1 year option
print(f"\nImpact of Rate Changes (1-Year ATM Call):")
print(f"  {'Interest Rate':<18} {'Call Price':<15} {'Price Change':<15} {'% Change':<15}")
print("-" * 70)

base_rate = 0.05
base_price = black_scholes_price(S, K, T, base_rate, sigma, 'call')

for rate_change in [-0.02, -0.01, 0, 0.01, 0.02]:
    new_rate = base_rate + rate_change
    new_price = black_scholes_price(S, K, T, new_rate, sigma, 'call')
    price_change = new_price - base_price
    pct_change = (price_change / base_price) * 100
    
    print(f"  {new_rate:.2%:<17} ${new_price:>13.4f} ${price_change:>13.4f} {pct_change:>13.2f}%")

# Calculate rho for 1-year option
rho_call_1y = calculate_rho(S, K, 1.0, r, sigma, 'call')
rho_put_1y = calculate_rho(S, K, 1.0, r, sigma, 'put')

print(f"\nRho Analysis (1-Year ATM Options):")
print(f"  Call Rho: {rho_call_1y:.6f}")
print(f"  Put Rho: {rho_put_1y:.6f}")

print(f"\nInterpretation:")
print(f"  • If rates increase 1% (5% → 6%):")
print(f"    - Call price increases by ${rho_call_1y:.4f}")
print(f"    - Put price decreases by ${abs(rho_put_1y):.4f}")
print(f"  • If rates increase 0.25% (5% → 5.25%):")
print(f"    - Call price increases by ${rho_call_1y * 0.25:.4f}")
print(f"    - Put price decreases by ${abs(rho_put_1y) * 0.25:.4f}")

# Historical perspective: 2022 rate hikes
print(f"\nHistorical Example: 2022 Fed Rate Hikes")
print(f"  Scenario: Fed raises rates from 0.25% to 4.25% (400 bps)")

rate_initial = 0.0025
rate_final = 0.0425
rate_change = rate_final - rate_initial

call_price_low_rate = black_scholes_price(S, K, T, rate_initial, sigma, 'call')
call_price_high_rate = black_scholes_price(S, K, T, rate_final, sigma, 'call')
actual_change = call_price_high_rate - call_price_low_rate

rho_estimate = rho_call_1y * (rate_change * 100)

print(f"  Initial Call Price (r=0.25%): ${call_price_low_rate:.4f}")
print(f"  Final Call Price (r=4.25%): ${call_price_high_rate:.4f}")
print(f"  Actual Price Change: ${actual_change:.4f}")
print(f"  Rho-Based Estimate: ${rho_estimate:.4f}")
print(f"  (Small difference due to non-linearity)")

# Portfolio rho
n_contracts = 100
contract_size = 100
total_rho = rho_call_1y * n_contracts * contract_size

print(f"\nPortfolio Rho (Long {n_contracts} 1-Year Call Contracts):")
print(f"  Total Rho: ${total_rho:,.2f}")
print(f"  For 1% rate increase: Portfolio gains ${total_rho:,.2f}")
print(f"  For 0.25% rate increase: Portfolio gains ${total_rho * 0.25:,.2f}")

print("\n" + "=" * 80)
print('[OK] Rho calculation complete')

In [ ]:
# Visualization: Rho Across Parameters

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

S = 100
K = 100
r = 0.05
sigma = 0.25

# Plot 1: Rho vs Stock Price
S_range = np.linspace(70, 130, 100)
T = 1.0

rhos_call = [calculate_rho(S_val, K, T, r, sigma, 'call') for S_val in S_range]
rhos_put = [calculate_rho(S_val, K, T, r, sigma, 'put') for S_val in S_range]

ax1.plot(S_range, rhos_call, 'b-', linewidth=2, label='Call Rho')
ax1.plot(S_range, rhos_put, 'r-', linewidth=2, label='Put Rho')
ax1.axvline(x=K, color='green', linestyle='--', linewidth=1.5, label=f'Strike K=${K}')
ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax1.set_xlabel('Stock Price ($)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Rho ($ per 1% rate change)', fontsize=12, fontweight='bold')
ax1.set_title('Rho vs Stock Price (T=1y)', fontsize=13, fontweight='bold')
ax1.legend(loc='best', fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 2: Rho vs Time to Expiration
T_range = np.linspace(0.05, 2.0, 100)
S = 100

rhos_time_call = [calculate_rho(S, K, T_val, r, sigma, 'call') for T_val in T_range]
rhos_time_put = [calculate_rho(S, K, T_val, r, sigma, 'put') for T_val in T_range]

ax2.plot(T_range, rhos_time_call, 'b-', linewidth=2, label='Call Rho')
ax2.plot(T_range, rhos_time_put, 'r-', linewidth=2, label='Put Rho')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.set_xlabel('Time to Expiration (years)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Rho ($ per 1% rate change)', fontsize=12, fontweight='bold')
ax2.set_title('ATM Rho vs Time to Expiration', fontsize=13, fontweight='bold')
ax2.legend(loc='best', fontsize=10)
ax2.grid(True, alpha=0.3)

# Plot 3: Option Price vs Interest Rate
r_range = np.linspace(0.01, 0.10, 100)
T = 1.0

prices_call = [black_scholes_price(S, K, T, r_val, sigma, 'call') for r_val in r_range]
prices_put = [black_scholes_price(S, K, T, r_val, sigma, 'put') for r_val in r_range]

ax3.plot(r_range * 100, prices_call, 'b-', linewidth=2, label='Call Price')
ax3.plot(r_range * 100, prices_put, 'r-', linewidth=2, label='Put Price')
ax3.axvline(x=r*100, color='green', linestyle='--', linewidth=1.5, label=f'Current r={r:.1%}')
ax3.set_xlabel('Interest Rate (%)', fontsize=12, fontweight='bold')
ax3.set_ylabel('Option Price ($)', fontsize=12, fontweight='bold')
ax3.set_title('ATM Option Price vs Interest Rate (T=1y)', fontsize=13, fontweight='bold')
ax3.legend(loc='best', fontsize=10)
ax3.grid(True, alpha=0.3)

# Plot 4: Rho vs Strike (for different expirations)
K_range = np.linspace(80, 120, 100)
S = 100
T_scenarios = [0.25, 1.0, 2.0]

for T_val in T_scenarios:
    rhos_strike = [calculate_rho(S, K_val, T_val, r, sigma, 'call') for K_val in K_range]
    ax4.plot(K_range, rhos_strike, linewidth=2, label=f'T={T_val:.2f}y')

ax4.axvline(x=S, color='red', linestyle='--', linewidth=1.5, label=f'Stock S=${S}')
ax4.set_xlabel('Strike Price ($)', fontsize=12, fontweight='bold')
ax4.set_ylabel('Call Rho ($ per 1% rate change)', fontsize=12, fontweight='bold')
ax4.set_title('Call Rho vs Strike for Different Expirations', fontsize=13, fontweight='bold')
ax4.legend(loc='best', fontsize=10)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('[OK] Rho visualization complete')

### Practical Example: LEAPS Sensitivity to Rate Changes

Let's analyze how long-dated options (LEAPS) are more sensitive to interest rate changes.

In [ ]:
# Practical Example: LEAPS vs Short-Term Options Rho Exposure

print("=" * 80)
print("RHO PRACTICAL EXAMPLE: LEAPS VS SHORT-TERM OPTIONS")
print("=" * 80)

S = 100
K = 100
r_initial = 0.02  # Low rate environment
r_final = 0.05    # After rate hikes
sigma = 0.25

print(f"\nScenario: Central Bank Rate Hikes")
print(f"  Initial Rate: {r_initial:.2%}")
print(f"  Final Rate: {r_final:.2%}")
print(f"  Rate Increase: {(r_final - r_initial)*100:.0f} basis points")

# Compare short-term vs long-term options
options = [
    (30/365, "1-Month Call"),
    (90/365, "3-Month Call"),
    (180/365, "6-Month Call"),
    (1.0, "1-Year Call (LEAPS)"),
    (2.0, "2-Year Call (LEAPS)")
]

print(f"\n{'Option Type':<22} {'Price @ 2%':<15} {'Price @ 5%':<15} {'Change':<15} {'Rho':<12}")
print("-" * 85)

for T, label in options:
    price_low = black_scholes_price(S, K, T, r_initial, sigma, 'call')
    price_high = black_scholes_price(S, K, T, r_final, sigma, 'call')
    price_change = price_high - price_low
    rho = calculate_rho(S, K, T, r_initial, sigma, 'call')
    
    print(f"{label:<22} ${price_low:>13.4f} ${price_high:>13.4f} ${price_change:>13.4f} {rho:>11.6f}")

print(f"\nKey Observations:")
print(f"  • Short-term options: Minimal rho impact (< $0.02 change)")
print(f"  • 1-Year LEAPS: Moderate impact (~${black_scholes_price(S, K, 1.0, r_final, sigma, 'call') - black_scholes_price(S, K, 1.0, r_initial, sigma, 'call'):.2f})")
print(f"  • 2-Year LEAPS: Significant impact (~${black_scholes_price(S, K, 2.0, r_final, sigma, 'call') - black_scholes_price(S, K, 2.0, r_initial, sigma, 'call'):.2f})")

# Portfolio example: 100 contracts of 2-year LEAPS
n_contracts = 100
contract_size = 100
T_leaps = 2.0

rho_leaps = calculate_rho(S, K, T_leaps, r_initial, sigma, 'call')
total_rho = rho_leaps * n_contracts * contract_size

price_leaps_low = black_scholes_price(S, K, T_leaps, r_initial, sigma, 'call')
price_leaps_high = black_scholes_price(S, K, T_leaps, r_final, sigma, 'call')

portfolio_value_low = price_leaps_low * n_contracts * contract_size
portfolio_value_high = price_leaps_high * n_contracts * contract_size
portfolio_pnl = portfolio_value_high - portfolio_value_low

print(f"\nPortfolio Impact ({n_contracts} Contracts of 2-Year LEAPS):")
print(f"  Initial Portfolio Value (r=2%): ${portfolio_value_low:,.2f}")
print(f"  Final Portfolio Value (r=5%): ${portfolio_value_high:,.2f}")
print(f"  P&L from Rate Change: ${portfolio_pnl:,.2f}")
print(f"  Return: {(portfolio_pnl / portfolio_value_low) * 100:.2f}%")

print(f"\n  Portfolio Rho: ${total_rho:,.2f}")
print(f"  Rho-based P&L estimate: ${total_rho * (r_final - r_initial) * 100:,.2f}")

# Hedging rho exposure
print(f"\nHedging Rho Exposure:")
print(f"  Problem: Long LEAPS portfolio has rho = ${total_rho:,.2f}")
print(f"  Solution: Can't directly hedge rho with stock (stock has no rho)")
print(f"  Options:")
print(f"    1. Short shorter-dated calls to partially offset")
print(f"    2. Use interest rate derivatives (futures, swaps)")
print(f"    3. Accept rho as part of overall portfolio risk")

# Real-world context
print(f"\nReal-World Context:")
print(f"  • 2020-2021: Near-zero rates, rho barely mattered")
print(f"  • 2022-2023: Rapid rate hikes, LEAPS traders saw unexpected gains on calls")
print(f"  • LEAPS portfolio management requires monitoring rate expectations")
print(f"  • Fed policy changes can materially impact LEAPS portfolios")

print("\n" + "=" * 80)
print('[OK] LEAPS rho example complete')

## 7. Portfolio Greeks

### Theory

**Portfolio Greeks** aggregate risk metrics across multiple positions:

$$
\text{Greek}_{\text{portfolio}} = \sum_{i=1}^{n} w_i \cdot \text{Greek}_i
$$

where $w_i$ is the quantity (with sign) of instrument $i$.

### Portfolio Risk Management

**Multi-Position Analysis**:
- Calculate Greeks for each individual position
- Aggregate to get total portfolio exposure
- Monitor against risk limits
- Rebalance when limits are breached

**Sign Conventions**:
- Long positions: Use positive quantities
- Short positions: Use negative quantities
- Net exposure: Algebraic sum

### Common Portfolio Strategies

1. **Delta-Neutral Portfolio**: $\Delta_{\text{port}} = 0$
   - No directional exposure
   - Profits from gamma/vega/theta

2. **Vega-Neutral Portfolio**: $\nu_{\text{port}} = 0$
   - No volatility exposure
   - Isolate delta or theta exposure

3. **Gamma-Scalping**: Positive gamma, negative theta
   - Profit from frequent rehedging if realized vol > implied vol

### Risk Limits

Trading desks typically set limits on:
- **Delta**: Maximum directional exposure
- **Vega**: Maximum volatility exposure  
- **Gamma**: Maximum convexity exposure
- **Theta**: Maximum daily time decay

### Mathematical Formulation

For a portfolio with $n$ positions:

$$
\Delta_{\text{port}} = \sum_{i=1}^{n} q_i \cdot \Delta_i + q_{\text{stock}} \cdot 1
$$

where:
- $q_i$ = quantity of option $i$ (negative for shorts)
- $\Delta_i$ = delta of option $i$
- $q_{\text{stock}}$ = stock position (shares)

**Similarly for other Greeks**:

$$
\Gamma_{\text{port}} = \sum_{i=1}^{n} q_i \cdot \Gamma_i
$$

$$
\nu_{\text{port}} = \sum_{i=1}^{n} q_i \cdot \nu_i
$$

$$
\Theta_{\text{port}} = \sum_{i=1}^{n} q_i \cdot \Theta_i
$$

$$
\rho_{\text{port}} = \sum_{i=1}^{n} q_i \cdot \rho_i
$$

Note: Stock positions have $\Delta = 1$, $\Gamma = 0$, $\nu = 0$, $\Theta = 0$, $\rho = 0$.

In [ ]:
# Portfolio Greeks Calculator

class OptionPosition:
    """Represents a single option position with its Greeks."""
    
    def __init__(self, S, K, T, r, sigma, option_type, quantity, contract_size=100):
        """
        Initialize option position.
        
        Parameters
        ----------
        S : float
            Stock price
        K : float
            Strike price
        T : float
            Time to expiration (years)
        r : float
            Risk-free rate
        sigma : float
            Volatility
        option_type : str
            'call' or 'put'
        quantity : int
            Number of contracts (negative for short)
        contract_size : int
            Shares per contract (default 100)
        """
        self.S = S
        self.K = K
        self.T = T
        self.r = r
        self.sigma = sigma
        self.option_type = option_type
        self.quantity = quantity
        self.contract_size = contract_size
        
        # Calculate Greeks
        self.price = black_scholes_price(S, K, T, r, sigma, option_type)
        self.delta = calculate_delta(S, K, T, r, sigma, option_type)
        self.gamma = calculate_gamma(S, K, T, r, sigma)
        self.vega = calculate_vega(S, K, T, r, sigma)
        self.theta = calculate_theta(S, K, T, r, sigma, option_type)
        self.rho = calculate_rho(S, K, T, r, sigma, option_type)
        
    def portfolio_greeks(self):
        """Calculate portfolio-level Greeks for this position."""
        mult = self.quantity * self.contract_size
        return {
            'value': self.price * mult,
            'delta': self.delta * mult,
            'gamma': self.gamma * mult,
            'vega': self.vega * mult,
            'theta': self.theta * mult,
            'rho': self.rho * mult
        }


class PortfolioGreeks:
    """Portfolio-level Greeks calculator."""
    
    def __init__(self):
        self.positions = []
        self.stock_position = 0
        
    def add_option(self, S, K, T, r, sigma, option_type, quantity, contract_size=100):
        """Add option position to portfolio."""
        position = OptionPosition(S, K, T, r, sigma, option_type, quantity, contract_size)
        self.positions.append(position)
        
    def add_stock(self, quantity):
        """Add stock position (in shares)."""
        self.stock_position += quantity
        
    def calculate_portfolio_greeks(self):
        """Calculate aggregate portfolio Greeks."""
        portfolio = {
            'value': 0,
            'delta': self.stock_position,  # Stock has delta = 1
            'gamma': 0,
            'vega': 0,
            'theta': 0,
            'rho': 0
        }
        
        for position in self.positions:
            greeks = position.portfolio_greeks()
            for key in portfolio:
                portfolio[key] += greeks[key]
        
        return portfolio
    
    def display_summary(self):
        """Display portfolio summary."""
        print("Portfolio Summary:")
        print("=" * 80)
        
        if self.stock_position != 0:
            print(f"\nStock Position: {self.stock_position:,.0f} shares")
        
        print(f"\nOption Positions:")
        print(f"  {'Type':<8} {'Strike':<10} {'Expiry':<12} {'Quantity':<12} {'Delta':<12} {'Price':<12}")
        print("-" * 80)
        
        for pos in self.positions:
            exp_days = pos.T * 365
            qty_str = f"{pos.quantity:+d} contracts"
            print(f"  {pos.option_type.upper():<8} ${pos.K:<9.2f} {exp_days:<11.0f}d {qty_str:<12} "
                  f"{pos.delta:>11.4f} ${pos.price:>11.4f}")
        
        portfolio = self.calculate_portfolio_greeks()
        
        print(f"\nPortfolio Greeks:")
        print("-" * 80)
        print(f"  Portfolio Value: ${portfolio['value']:,.2f}")
        print(f"  Delta:           {portfolio['delta']:,.2f}")
        print(f"  Gamma:           {portfolio['gamma']:.6f}")
        print(f"  Vega:            ${portfolio['vega']:,.2f}")
        print(f"  Theta:           ${portfolio['theta']:,.2f} per day")
        print(f"  Rho:             ${portfolio['rho']:,.2f}")
        print("=" * 80)


# Example: Multi-Position Portfolio
print("=" * 80)
print("PORTFOLIO GREEKS: MULTI-POSITION EXAMPLE")
print("=" * 80)

S = 100
r = 0.05
sigma = 0.25

# Create portfolio
portfolio = PortfolioGreeks()

# Position 1: Long 10 ATM calls (3 months)
portfolio.add_option(S, K=100, T=0.25, r=r, sigma=sigma, option_type='call', quantity=10)

# Position 2: Short 5 OTM calls (3 months)
portfolio.add_option(S, K=110, T=0.25, r=r, sigma=sigma, option_type='call', quantity=-5)

# Position 3: Long 10 ATM puts (3 months)  
portfolio.add_option(S, K=100, T=0.25, r=r, sigma=sigma, option_type='put', quantity=10)

# Position 4: Delta hedge with stock
# First calculate option delta
temp_greeks = portfolio.calculate_portfolio_greeks()
option_delta = temp_greeks['delta']
portfolio.add_stock(-int(option_delta))  # Short shares to hedge

# Display full summary
portfolio.display_summary()

print(f"\nInterpretation:")
print(f"  • Near-zero delta: Portfolio is approximately delta-neutral")
print(f"  • Positive gamma: Benefits from large price moves")
print(f"  • Positive vega: Benefits from volatility increases")
print(f"  • Negative theta: Loses value over time (cost of gamma)")

print("\n" + "=" * 80)
print('[OK] Portfolio Greeks calculation complete')

In [ ]:
# Visualization: Portfolio Greeks Analysis

# Create a more complex portfolio for visualization
S = 100
r = 0.05
sigma = 0.25

# Iron Condor strategy
iron_condor = PortfolioGreeks()
iron_condor.add_option(S, K=95, T=0.25, r=r, sigma=sigma, option_type='put', quantity=-10)   # Short put
iron_condor.add_option(S, K=90, T=0.25, r=r, sigma=sigma, option_type='put', quantity=10)    # Long put
iron_condor.add_option(S, K=105, T=0.25, r=r, sigma=sigma, option_type='call', quantity=-10) # Short call
iron_condor.add_option(S, K=110, T=0.25, r=r, sigma=sigma, option_type='call', quantity=10)  # Long call

# Calculate P&L across stock price range
S_range = np.linspace(80, 120, 100)
pnl_profile = []
delta_profile = []
gamma_profile = []
theta_profile = []

initial_greeks = iron_condor.calculate_portfolio_greeks()
initial_value = initial_greeks['value']

for S_test in S_range:
    test_portfolio = PortfolioGreeks()
    test_portfolio.add_option(S_test, K=95, T=0.25, r=r, sigma=sigma, option_type='put', quantity=-10)
    test_portfolio.add_option(S_test, K=90, T=0.25, r=r, sigma=sigma, option_type='put', quantity=10)
    test_portfolio.add_option(S_test, K=105, T=0.25, r=r, sigma=sigma, option_type='call', quantity=-10)
    test_portfolio.add_option(S_test, K=110, T=0.25, r=r, sigma=sigma, option_type='call', quantity=10)
    
    greeks = test_portfolio.calculate_portfolio_greeks()
    pnl = greeks['value'] - initial_value
    pnl_profile.append(pnl)
    delta_profile.append(greeks['delta'])
    gamma_profile.append(greeks['gamma'])
    theta_profile.append(greeks['theta'])

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: P&L Profile
ax1.plot(S_range, pnl_profile, 'b-', linewidth=2.5)
ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax1.axvline(x=S, color='red', linestyle='--', linewidth=1.5, label=f'Current S=${S}')
ax1.fill_between(S_range, 0, pnl_profile, where=np.array(pnl_profile) > 0, alpha=0.3, color='green', label='Profit')
ax1.fill_between(S_range, 0, pnl_profile, where=np.array(pnl_profile) < 0, alpha=0.3, color='red', label='Loss')
ax1.set_xlabel('Stock Price ($)', fontsize=12, fontweight='bold')
ax1.set_ylabel('P&L ($)', fontsize=12, fontweight='bold')
ax1.set_title('Iron Condor P&L Profile', fontsize=13, fontweight='bold')
ax1.legend(loc='best', fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 2: Delta Profile
ax2.plot(S_range, delta_profile, 'purple', linewidth=2.5)
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.axvline(x=S, color='red', linestyle='--', linewidth=1.5)
ax2.set_xlabel('Stock Price ($)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Portfolio Delta', fontsize=12, fontweight='bold')
ax2.set_title('Portfolio Delta vs Stock Price', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Plot 3: Gamma Profile
ax3.plot(S_range, gamma_profile, 'green', linewidth=2.5)
ax3.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax3.axvline(x=S, color='red', linestyle='--', linewidth=1.5)
ax3.set_xlabel('Stock Price ($)', fontsize=12, fontweight='bold')
ax3.set_ylabel('Portfolio Gamma', fontsize=12, fontweight='bold')
ax3.set_title('Portfolio Gamma vs Stock Price', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3)

# Plot 4: Theta Profile
ax4.plot(S_range, theta_profile, 'orange', linewidth=2.5)
ax4.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax4.axvline(x=S, color='red', linestyle='--', linewidth=1.5)
ax4.set_xlabel('Stock Price ($)', fontsize=12, fontweight='bold')
ax4.set_ylabel('Portfolio Theta ($ per day)', fontsize=12, fontweight='bold')
ax4.set_title('Portfolio Theta vs Stock Price', fontsize=13, fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('[OK] Portfolio Greeks visualization complete')

### Practical Example: Risk Monitoring Dashboard

Portfolio managers use Greeks dashboards to monitor and manage risk exposure in real-time.

In [ ]:
# Practical Example: Trading Desk Risk Dashboard

print("=" * 90)
print("RISK MANAGEMENT DASHBOARD: TRADING DESK PORTFOLIO")
print("=" * 90)

S = 100
r = 0.05
sigma = 0.25

# Simulate a trading desk with multiple strategies
trading_book = PortfolioGreeks()

# Strategy 1: Market Making (delta-hedged)
trading_book.add_option(S, K=100, T=0.08, r=r, sigma=sigma, option_type='call', quantity=-50)   # Short calls
trading_book.add_option(S, K=100, T=0.08, r=r, sigma=sigma, option_type='put', quantity=-50)    # Short puts
trading_book.add_stock(50 * 100 * 0.5)  # Approximate delta hedge

# Strategy 2: Earnings volatility trade
trading_book.add_option(S, K=100, T=0.05, r=r, sigma=0.35, option_type='call', quantity=20)  # Long straddle
trading_book.add_option(S, K=100, T=0.05, r=r, sigma=0.35, option_type='put', quantity=20)

# Strategy 3: Calendar spread
trading_book.add_option(S, K=105, T=0.16, r=r, sigma=sigma, option_type='call', quantity=-30)  # Short near
trading_book.add_option(S, K=105, T=0.5, r=r, sigma=sigma, option_type='call', quantity=30)    # Long far

# Calculate portfolio Greeks
greeks = trading_book.calculate_portfolio_greeks()

# Define risk limits (example)
risk_limits = {
    'delta': 10000,        # Max delta exposure
    'gamma': 50,           # Max gamma
    'vega': 50000,         # Max vega (per 1% vol change)
    'theta': -5000         # Max theta decay per day
}

print(f"\nRISK METRICS")
print("-" * 90)
print(f"  {'Metric':<20} {'Value':<25} {'Limit':<20} {'% of Limit':<15} {'Status':<10}")
print("-" * 90)

metrics = [
    ('Portfolio Value', f"${greeks['value']:,.2f}", 'N/A', 'N/A', 'INFO'),
    ('Delta', f"{greeks['delta']:,.2f} shares", f"+/- {risk_limits['delta']:,}", 
     f"{abs(greeks['delta'])/risk_limits['delta']*100:.1f}%", 
     'OK' if abs(greeks['delta']) < risk_limits['delta'] else 'WARNING'),
    ('Gamma', f"{greeks['gamma']:.4f}", f"+/- {risk_limits['gamma']}", 
     f"{abs(greeks['gamma'])/risk_limits['gamma']*100:.1f}%",
     'OK' if abs(greeks['gamma']) < risk_limits['gamma'] else 'WARNING'),
    ('Vega', f"${greeks['vega']:,.2f}", f"+/- ${risk_limits['vega']:,}", 
     f"{abs(greeks['vega'])/risk_limits['vega']*100:.1f}%",
     'OK' if abs(greeks['vega']) < risk_limits['vega'] else 'WARNING'),
    ('Theta', f"${greeks['theta']:,.2f} /day", f"> ${risk_limits['theta']:,}", 
     f"{greeks['theta']/risk_limits['theta']*100:.1f}%",
     'OK' if greeks['theta'] > risk_limits['theta'] else 'WARNING'),
    ('Rho', f"${greeks['rho']:,.2f}", 'N/A', 'N/A', 'INFO')
]

for metric, value, limit, pct, status in metrics:
    status_symbol = '[OK]' if status == 'OK' else '[!]' if status == 'WARNING' else '[i]'
    print(f"  {metric:<20} {value:<25} {limit:<20} {pct:<15} {status_symbol:<10}")

# Scenario Analysis
print(f"\nSCENARIO ANALYSIS")
print("-" * 90)

scenarios = [
    ("Base Case", S, sigma, 0),
    ("Stock +5%", S * 1.05, sigma, 0),
    ("Stock -5%", S * 0.95, sigma, 0),
    ("Vol +5%", S, sigma + 0.05, 0),
    ("5 Days Pass", S, sigma, 5)
]

print(f"  {'Scenario':<20} {'Portfolio Value':<20} {'Change':<20} {'% Change':<15}")
print("-" * 90)

base_value = greeks['value']

for scenario_name, S_scenario, sigma_scenario, days_passed in scenarios:
    # Recalculate portfolio with scenario
    test_book = PortfolioGreeks()
    
    # Adjust time for all positions
    T_adjust = lambda T: max(T - days_passed/365, 0.001)
    
    test_book.add_option(S_scenario, K=100, T=T_adjust(0.08), r=r, sigma=sigma_scenario, 
                        option_type='call', quantity=-50)
    test_book.add_option(S_scenario, K=100, T=T_adjust(0.08), r=r, sigma=sigma_scenario, 
                        option_type='put', quantity=-50)
    test_book.add_stock(50 * 100 * 0.5)
    
    test_book.add_option(S_scenario, K=100, T=T_adjust(0.05), r=r, sigma=0.35 if sigma_scenario == sigma else sigma_scenario + 0.1, 
                        option_type='call', quantity=20)
    test_book.add_option(S_scenario, K=100, T=T_adjust(0.05), r=r, sigma=0.35 if sigma_scenario == sigma else sigma_scenario + 0.1, 
                        option_type='put', quantity=20)
    
    test_book.add_option(S_scenario, K=105, T=T_adjust(0.16), r=r, sigma=sigma_scenario, 
                        option_type='call', quantity=-30)
    test_book.add_option(S_scenario, K=105, T=T_adjust(0.5), r=r, sigma=sigma_scenario, 
                        option_type='call', quantity=30)
    
    test_greeks = test_book.calculate_portfolio_greeks()
    change = test_greeks['value'] - base_value
    pct_change = (change / base_value) * 100
    
    print(f"  {scenario_name:<20} ${test_greeks['value']:>18,.2f} ${change:>18,.2f} {pct_change:>13.2f}%")

print(f"\nRISK MANAGEMENT ACTIONS")
print("-" * 90)
print(f"  • Delta: {abs(greeks['delta']):.0f} < limit of {risk_limits['delta']:,} → No action needed")
print(f"  • Gamma: Within limits, monitor for large moves")
print(f"  • Vega: {abs(greeks['vega']):.0f} vs limit {risk_limits['vega']:,} → Monitor implied vol changes")
print(f"  • Theta: Collecting ${abs(greeks['theta']):,.2f}/day from short gamma positions")

print("\n" + "=" * 90)
print('[OK] Risk dashboard complete')

## Comprehensive Case Study: Managing a Multi-Strategy Options Portfolio

Let's combine all concepts in a comprehensive real-world scenario: a volatility arbitrage fund managing multiple strategies with strict risk controls.

In [ ]:
# Comprehensive Case Study: Volatility Arbitrage Fund

print("=" * 100)
print("CASE STUDY: VOLATILITY ARBITRAGE FUND - COMPREHENSIVE GREEKS MANAGEMENT")
print("=" * 100)

print("\nSITUATION:")
print("-" * 100)
print("A volatility arbitrage fund manages a $10M portfolio trading options on a single underlying.")
print("The fund believes implied volatility is too high and will decline over the next month.")
print("Strategy: Sell volatility (short options) while maintaining delta-neutral positioning.\n")

# Market conditions
S = 100
r = 0.05
implied_vol = 0.35  # High implied volatility
realized_vol = 0.25  # Fund's estimate of actual volatility

print("MARKET CONDITIONS:")
print("-" * 100)
print(f"  Stock Price: ${S:.2f}")
print(f"  Implied Volatility: {implied_vol:.1%} (market pricing)")
print(f"  Estimated Realized Vol: {realized_vol:.1%} (fund's forecast)")
print(f"  Risk-Free Rate: {r:.1%}")
print(f"  Opportunity: Sell overpriced options, profit from vol decline\n")

# Build the portfolio
fund = PortfolioGreeks()

# Strategy 1: Sell ATM straddles (1 month) - Primary vol short
fund.add_option(S, K=100, T=1/12, r=r, sigma=implied_vol, option_type='call', quantity=-100)
fund.add_option(S, K=100, T=1/12, r=r, sigma=implied_vol, option_type='put', quantity=-100)

# Strategy 2: Sell near-term strangles (2 weeks) - Additional premium
fund.add_option(S, K=105, T=2/52, r=r, sigma=implied_vol, option_type='call', quantity=-150)
fund.add_option(S, K=95, T=2/52, r=r, sigma=implied_vol, option_type='put', quantity=-150)

# Strategy 3: Buy longer-dated protection (3 months) - Tail risk hedge
fund.add_option(S, K=110, T=3/12, r=r, sigma=implied_vol, option_type='call', quantity=50)
fund.add_option(S, K=90, T=3/12, r=r, sigma=implied_vol, option_type='put', quantity=50)

# Initial Greeks
initial_greeks = fund.calculate_portfolio_greeks()

# Delta hedge with stock
shares_to_hedge = -initial_greeks['delta']
fund.add_stock(shares_to_hedge)

# Recalculate after hedge
hedged_greeks = fund.calculate_portfolio_greeks()

print("INITIAL PORTFOLIO STRUCTURE:")
print("-" * 100)
fund.display_summary()

print("\nSTRATEGIC ANALYSIS:")
print("-" * 100)
premium_collected = abs(initial_greeks['value'])
print(f"  Net Premium Collected: ${premium_collected:,.2f}")
print(f"  Portfolio Delta (after hedge): {hedged_greeks['delta']:.2f} (delta-neutral)")
print(f"  Portfolio Gamma: {hedged_greeks['gamma']:.6f} (negative - short gamma)")
print(f"  Portfolio Vega: ${hedged_greeks['vega']:,.2f} (negative - short vol)")
print(f"  Portfolio Theta: ${hedged_greeks['theta']:,.2f} per day (positive - collecting decay)")
print(f"\n  Risk/Reward Profile:")
print(f"    • Profit if: Volatility declines and/or time passes without large moves")
print(f"    • Loss if: Volatility spikes or large price swings occur")
print(f"    • Theta collection: ${hedged_greeks['theta']:,.2f}/day × 30 days = ${hedged_greeks['theta'] * 30:,.2f}")

# Scenario 1: Volatility declines as expected
print("\n" + "=" * 100)
print("SCENARIO 1: SUCCESSFUL TRADE - VOLATILITY DECLINES")
print("=" * 100)

days_passed = 10
new_vol = 0.28  # Vol declined from 35% to 28%

# Rebuild portfolio with new conditions
fund_scenario1 = PortfolioGreeks()
fund_scenario1.add_option(S, K=100, T=(1/12 - days_passed/365), r=r, sigma=new_vol, 
                         option_type='call', quantity=-100)
fund_scenario1.add_option(S, K=100, T=(1/12 - days_passed/365), r=r, sigma=new_vol, 
                         option_type='put', quantity=-100)
fund_scenario1.add_option(S, K=105, T=(2/52 - days_passed/365), r=r, sigma=new_vol, 
                         option_type='call', quantity=-150)
fund_scenario1.add_option(S, K=95, T=(2/52 - days_passed/365), r=r, sigma=new_vol, 
                         option_type='put', quantity=-150)
fund_scenario1.add_option(S, K=110, T=(3/12 - days_passed/365), r=r, sigma=new_vol, 
                         option_type='call', quantity=50)
fund_scenario1.add_option(S, K=90, T=(3/12 - days_passed/365), r=r, sigma=new_vol, 
                         option_type='put', quantity=50)
fund_scenario1.add_stock(shares_to_hedge)

greeks_scenario1 = fund_scenario1.calculate_portfolio_greeks()
pnl_scenario1 = hedged_greeks['value'] - greeks_scenario1['value']

print(f"\nAfter {days_passed} days:")
print(f"  Stock Price: ${S:.2f} (unchanged)")
print(f"  Implied Volatility: {new_vol:.1%} (declined from {implied_vol:.1%})")
print(f"\nP&L Breakdown:")
print(f"  Vega P&L: ${(implied_vol - new_vol) * 100 * hedged_greeks['vega']:,.2f}")
print(f"  Theta P&L: ${hedged_greeks['theta'] * days_passed:,.2f}")
print(f"  Total P&L: ${pnl_scenario1:,.2f}")
print(f"  Return: {(pnl_scenario1 / premium_collected) * 100:.2f}% in {days_passed} days")

# Scenario 2: Adverse move - volatility spike
print("\n" + "=" * 100)
print("SCENARIO 2: ADVERSE MOVE - VOLATILITY SPIKE")
print("=" * 100)

days_passed_2 = 5
new_vol_spike = 0.45  # Vol spikes to 45%
S_move = 95  # Stock also drops 5%

fund_scenario2 = PortfolioGreeks()
fund_scenario2.add_option(S_move, K=100, T=(1/12 - days_passed_2/365), r=r, sigma=new_vol_spike, 
                         option_type='call', quantity=-100)
fund_scenario2.add_option(S_move, K=100, T=(1/12 - days_passed_2/365), r=r, sigma=new_vol_spike, 
                         option_type='put', quantity=-100)
fund_scenario2.add_option(S_move, K=105, T=(2/52 - days_passed_2/365), r=r, sigma=new_vol_spike, 
                         option_type='call', quantity=-150)
fund_scenario2.add_option(S_move, K=95, T=(2/52 - days_passed_2/365), r=r, sigma=new_vol_spike, 
                         option_type='put', quantity=-150)
fund_scenario2.add_option(S_move, K=110, T=(3/12 - days_passed_2/365), r=r, sigma=new_vol_spike, 
                         option_type='call', quantity=50)
fund_scenario2.add_option(S_move, K=90, T=(3/12 - days_passed_2/365), r=r, sigma=new_vol_spike, 
                         option_type='put', quantity=50)
fund_scenario2.add_stock(shares_to_hedge)

greeks_scenario2 = fund_scenario2.calculate_portfolio_greeks()
pnl_scenario2 = hedged_greeks['value'] - greeks_scenario2['value']

print(f"\nAfter {days_passed_2} days:")
print(f"  Stock Price: ${S_move:.2f} (down {((S_move/S)-1)*100:.1f}%)")
print(f"  Implied Volatility: {new_vol_spike:.1%} (spiked from {implied_vol:.1%})")
print(f"\nP&L Breakdown:")
print(f"  Vega P&L: ${(implied_vol - new_vol_spike) * 100 * hedged_greeks['vega']:,.2f} (loss from vol increase)")
print(f"  Delta P&L: ${hedged_greeks['delta'] * (S_move - S):,.2f} (minimal due to hedge)")
print(f"  Gamma P&L: Negative (short gamma hurts on large moves)")
print(f"  Total P&L: ${pnl_scenario2:,.2f}")
print(f"  Loss: {(pnl_scenario2 / premium_collected) * 100:.2f}%")
print(f"\nRisk Management Response:")
print(f"  • Long protection positions (110C, 90P) provide tail risk hedge")
print(f"  • Consider closing short-term positions to reduce vega exposure")
print(f"  • Rehedge delta as position has moved")

# Summary
print("\n" + "=" * 100)
print("KEY TAKEAWAYS FROM CASE STUDY:")
print("=" * 100)
print("\n1. GREEKS AS RISK MANAGEMENT TOOLS:")
print("   • Delta: Managed to near-zero through stock hedge")
print("   • Gamma: Negative (short gamma) - vulnerable to large moves")
print("   • Vega: Negative (short vol) - profits from vol decline")
print("   • Theta: Positive - collects time decay daily")

print("\n2. VOLATILITY TRADING:")
print("   • Selling overpriced options can be profitable if vol declines")
print("   • Requires accurate vol forecasting and risk management")
print("   • Negative gamma means rehedging costs money in volatile markets")

print("\n3. PORTFOLIO CONSTRUCTION:")
print("   • Multiple expiries diversify theta collection")
print("   • Long wings provide tail risk protection")
print("   • Delta hedging isolates vol exposure from directional risk")

print("\n4. REAL-WORLD CONSIDERATIONS:")
print("   • Transaction costs reduce profitability of frequent rehedging")
print("   • Bid-ask spreads matter when trading large size")
print("   • Greeks change over time - requires active monitoring")
print("   • Model risk: Black-Scholes assumptions may not hold")

print("\n" + "=" * 100)
print("[OK] Comprehensive case study complete")
print("=" * 100)

## Practice Exercises\n\nTest your understanding with these exercises:\n\n### Exercise 1\nDescription of exercise 1...\n\n### Exercise 2\nDescription of exercise 2...\n\n### Exercise 3\nDescription of exercise 3...

In [ ]:
# Your solution for Exercise 1



## Key Takeaways

1. **Delta** measures price sensitivity and serves as the hedge ratio for options
2. **Delta hedging** eliminates directional risk but requires continuous rebalancing
3. **Vega** measures volatility sensitivity - critical for volatility trading strategies
4. **Theta** quantifies time decay - the cost of holding long options or profit from short options
5. **Rho** measures interest rate sensitivity - important for long-dated options
6. **Portfolio Greeks** aggregate individual position risks for comprehensive risk management
7. **Greeks interact**: Theta-gamma trade-off, vega-gamma relationship, delta-neutral strategies

## Further Reading

- Hull, J.C. (2022). *Options, Futures, and Other Derivatives* (11th ed.)
- Taleb, N.N. (1997). *Dynamic Hedging: Managing Vanilla and Exotic Options*
- Wilmott, P. (2006). *Paul Wilmott on Quantitative Finance*
- Gatheral, J. (2006). *The Volatility Surface: A Practitioner's Guide*
- Sinclair, E. (2013). *Volatility Trading* (2nd ed.)

---## References### Academic Papers and Books1. Hull, J.C. (2021). *Options, Futures, and Other Derivatives (11th ed.)*. Pearson. Comprehensive derivatives textbook.2. Wilmott, P. (2006). *Paul Wilmott on Quantitative Finance*. Wiley. Three-volume set covering mathematical finance.3. Gatheral, J. (2006). *The Volatility Surface: A Practitioner's Guide*. Wiley. Essential volatility modeling reference.4. Andersen, T.G. et al. (2003). *Modeling and Forecasting Realized Volatility*. Econometrica, 71(2), 579-625.### Online Resources1. ***QuantLib Documentation*** - https://www.quantlib.org/ - Open-source quantitative finance library.2. ***Quantitative Finance on arXiv*** - https://arxiv.org/archive/q-fin - Latest research papers.3. ***Financial Python*** - https://github.com/quantopian - Quantopian lectures and tutorials.### Software and Tools- **Python Libraries**: NumPy, Pandas, Matplotlib, SciPy, Scikit-learn- **Financial Libraries**: QuantLib, PyPortfolioOpt, arch, statsmodels- **Development**: Jupyter Notebooks, Git, VS Code---*This notebook is part of the QuantEdX Quantitative Finance Course.**For questions or feedback, please contact: content@quantedx.com*